# 🏀 Category-Dominance Draft Engine v4 — 2026-27
## Swiss See Ballers · ESPN H2H Each-Category Salary-Cap Auction

**Draft: Oct 17, 2026 @ 20:00 · 14 teams · $200 cap · 13 spots (10 start) · 10 categories · max 4 C**

v3 asked *"which categories should I punt?"* and answered with a z-score plan. v4 re-checks everything v3 rested on
against data, and replaces the plan with a direct optimisation of the thing that decides a week:
**P(win ≥ 6 of 10 categories)**. What follows is a list of what was measured, because several v3 conclusions did
not survive.

### What was wrong in v3, and the evidence

| # | v3 assumption | measured here | consequence |
|---|---|---|---|
| 1 | Availability = mean of last two seasons' games, then ^1.35 | Cell 6B: next-season games played is predicted *worse* by that mean (RMSE 0.263) than by the same number for everyone (0.230). A 79-game player plays ~63 next year, not 79. | "Fade Giannis / Tatum / Sabonis for durability" was mostly noise; the spread was 2× too wide |
| 2 | Weekly simulator draws every box-score quantity as an independent normal | Cell 9: weekly team FT% sd 0.223 vs 0.047 in real games (4.7×), FG% 2.0×, PPM 1.2×; DD/BLK/STL means 3-15% too high | FG% and FT% became permanent coin-flips, so conceding them looked free and targeting them useless. Bundle ranking was biased toward punting them |
| 3 | Market = ESPN values stretched ×1.58 uniformly | ESPN's published curve is a ~10-team curve. Going to 14 teams lifts the *mid-tier* a lot and stars only 10-20% (Jokic ~$108, not $128 as in the uniform curve). The best plan **flips** between market models | A single "best plan" is not robust; it was an artifact of the price curve |
| 4 | Roster chosen by maximising a z-score sum for a fixed category set | Cell 11: optimising the real objective directly, at the same prices, in the same auction, wins far more weeks. The optimum is **broad** (≈7.4 categories/week), not a punt | The category *plan* is an output, not an input |
| 5 | Teammate-context factor applied by the **level** of the supporting cast | Walk-forward backtest (ad-hoc script, numbers in README.md, section "Befunde"): the **change** vs last year's cast explains 3.6× more variance; with both in the model the level coefficient is 0.00 | Jokic got +12% for a cast he already played with (double count) |
| 6 | Pool = players with ≥15 GP in 2025-26 | 7 injured returnees (Haliburton $57) and 5 ESPN-priced rookies were missing; a nominated one cannot be logged | Money leaving the room would go untracked and inflation would drift |
| 7 | "Robustness": my team drafts first from the full pool | Cell 11-12: contested auctions, sharper rivals, wrong market | The old robustness table could not fail |

**What was checked and kept:** the 70/30 season blend (backtest optimum is 0.75-0.8, flat), the age curve (empirical
age effect matches v3 except 35+, which is −9% not −20%), ESPN's position eligibility, the 4-Center cap handling.

### Method in one paragraph
Everything a lineup does in a week is a sum over its ten starters, so team means and variances are sums of per-player
terms and P(win a category) is a normal CDF. That makes a roster evaluable in microseconds
(`draft_app/engine.py`), validated here against a bootstrap of real games. With a fast, faithful objective, the
roster can be optimised against it directly (a Lagrange price on the budget, steepest-ascent swaps), and the
shadow price of money gives every player a **break-even bid**: the most I can pay and still end with a roster at
least as good as the best one without him. The live app re-solves this after every logged pick.

### Honest limits
* The projections are the model's belief. A backtest puts per-game forecast error at about half a cross-player
  standard deviation, which this notebook does not add to the weekly noise, so absolute win probabilities and
  records are optimistic. The comparisons between bidders are the finding.
* The rivals are simulated managers. Against rivals running the same optimiser the edge shrinks (Cell 12).
* ESPN's position eligibility and injury status are as of the cached pull; check both on draft day.

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ═══════════════════════════════════════════════════════════════════
import os, time, math, itertools, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from nba_api.stats.endpoints import leaguegamelog, commonplayerinfo, commonteamroster
from nba_api.stats.static import teams as static_teams

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

CACHE_DIR = 'cache'
os.makedirs(CACHE_DIR, exist_ok=True)
RNG_SEED = 20261017          # the draft date as master seed -> every result reproducible

print('✅ Imports OK  ·  pandas', pd.__version__, '· numpy', np.__version__)
import sys; sys.path.insert(0, 'draft_app')      # engine.py / auction_sim.py live next to the app

✅ Imports OK  ·  pandas 2.3.3 · numpy 2.2.6


## Cell 2 · League rules — the single source of truth
Every ESPN setting lives in one cell. Unchanged from v3 except the durability weight, which is now measured (Cell 6B).

In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — LEAGUE RULES  (single source of truth)
# ═══════════════════════════════════════════════════════════════════
# Transcribed from the ESPN settings page:
# fantasy.espn.com/basketball/league/settings?leagueId=1015252784&seasonId=2027
# If ESPN changes a setting, this cell is the ONLY place that needs editing.

LEAGUE_NAME    = 'Swiss See Ballers'
YOUR_TEAM_NAME = 'Janartuo Ninjas'
TEAM_NAMES = [
    'Nordstern Hawks', 'Brick City', 'Transalpes Trail Shredders',
    'Primitive Analytics', 'Janartuo Ninjas', 'Inaugural Champs',
    'Ulifortada', 'St. Wendelin Storm', 'Switchler to the moon',
    'Mobiliar Mullets', 'Load Management Slyrix', 'Undersized Problems',
    'Team 15', "Noah's Nasty Team",
]
NUM_TEAMS    = len(TEAM_NAMES)          # 14
SCORING_TYPE = 'H2H_EACH_CATEGORY'

# ── Categories: ESPN label -> internal key ───────────────────────────────────
# This league scores DD and PPM and does NOT score turnovers, so these 10 are the complete
# category set. PPM (points per minute) is rare in ESPN leagues and turns out to matter.
CATEGORY_MAP = {
    'FG%': 'FG_PCT', 'FT%': 'FT_PCT', '3PM': 'FG3M',  'REB': 'REB', 'AST': 'AST',
    'STL': 'STL',    'BLK': 'BLK',    'DD':  'DD',    'PTS': 'PTS', 'PPM': 'PTS_MIN',
}
CATS = list(CATEGORY_MAP.values())
LBL  = {v: k for k, v in CATEGORY_MAP.items()}          # 'FG3M' -> '3PM'
KEY  = dict(CATEGORY_MAP)                               # '3PM'  -> 'FG3M'
CATS_TO_WIN = 6                                         # 6 of 10 wins the matchup

# How each category is settled in a weekly matchup:
COUNTING_CATS = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'DD']   # summed over games
RATIO_CATS    = ['FG_PCT', 'FT_PCT', 'PTS_MIN']                     # team aggregates
assert set(COUNTING_CATS) | set(RATIO_CATS) == set(CATS)

# ── Draft ────────────────────────────────────────────────────────────────────
DRAFT_TYPE      = 'SALARY_CAP'
DRAFT_DATE      = pd.Timestamp('2026-10-17 20:00')
BUDGET_PER_TEAM = 200
MIN_BID         = 1
TOTAL_BUDGET    = NUM_TEAMS * BUDGET_PER_TEAM           # $2,800

# ── Roster ───────────────────────────────────────────────────────────────────
ROSTER_SIZE    = 13
BENCH_COUNT    = 3
IR_SLOTS       = 0
TOTAL_DRAFTED  = NUM_TEAMS * ROSTER_SIZE                # 182
ALL_POSITIONS  = ['PG', 'SG', 'SF', 'PF', 'C']
STARTER_SLOTS  = [
    ('PG', ['PG']), ('SG', ['SG']), ('SF', ['SF']), ('PF', ['PF']), ('C', ['C']),
    ('G',  ['PG', 'SG']), ('F', ['SF', 'PF']),
    ('UTIL', ALL_POSITIONS), ('UTIL', ALL_POSITIONS), ('UTIL', ALL_POSITIONS),
]
TOTAL_STARTERS    = len(STARTER_SLOTS)                  # 10
SLOT_MASKS        = [frozenset(a) for _, a in STARTER_SLOTS]
POSITION_MAXIMUMS = {'C': 4}                            # only Center is capped
MAX_CENTERS       = POSITION_MAXIMUMS['C']
assert ROSTER_SIZE == TOTAL_STARTERS + BENCH_COUNT + IR_SLOTS

# ── Lineups / acquisitions ───────────────────────────────────────────────────
LINEUP_CHANGES    = 'WEEKLY_LOCK_AT_FIRST_GAME'
LINEUP_PROTECTION = False
FAB_BUDGET        = 100
MATCHUP_ACQUISITION_LIMIT = 7

# ── Season ───────────────────────────────────────────────────────────────────
WEEKS_PER_MATCHUP       = 1
REGULAR_SEASON_MATCHUPS = 18
PLAYOFF_TEAMS           = 4
CATEGORY_TIE_BREAKER    = None      # "No tie breakers" -> an exact tie stays a tie
KEEPERS                 = False     # pure redraft: no future value in any price
GAMES_PER_WEEK          = 3.45      # NBA average team games in a 7-day matchup

# ── Durability weight ──────────────────────────────────────────────────────
# v3 derived an exponent of 1.35 from the roster rules (no IR, no lineup protection, weekly lock). v4 measured it
# instead (Cell 6B): production is LINEAR in games played, and games played is itself only weakly predictable
# from past games. So the weight on a player is availability^1, with availability CALIBRATED, not averaged.
DURABILITY_EXPONENT = 1.0

print(f'📋 {LEAGUE_NAME}: {NUM_TEAMS} teams · {SCORING_TYPE} · ${BUDGET_PER_TEAM} cap')
print(f'   Roster {ROSTER_SIZE} = {TOTAL_STARTERS} starters + {BENCH_COUNT} bench + {IR_SLOTS} IR'
      f'   |   max {MAX_CENTERS} C   |   {len(CATS)} cats, need {CATS_TO_WIN} to win')
print(f'   Counting cats (summed over games): {", ".join(LBL[c] for c in COUNTING_CATS)}')
print(f'   Ratio cats (team aggregate)      : {", ".join(LBL[c] for c in RATIO_CATS)}')
print('   Durability: linear weight on CALIBRATED availability (v3: exponent 1.35 on a raw two-season mean)')
print(f'   Draft: {DRAFT_DATE:%b %d, %Y @ %H:%M}')

📋 Swiss See Ballers: 14 teams · H2H_EACH_CATEGORY · $200 cap
   Roster 13 = 10 starters + 3 bench + 0 IR   |   max 4 C   |   10 cats, need 6 to win
   Counting cats (summed over games): PTS, REB, AST, STL, BLK, 3PM, DD
   Ratio cats (team aggregate)      : FG%, FT%, PPM
   Durability: linear weight on CALIBRATED availability (v3: exponent 1.35 on a raw two-season mean)
   Draft: Oct 17, 2026 @ 20:00


## Cells 3–5B · Data
Two seasons of game logs (blend 70/30), player metadata, current 2026-27 rosters and ESPN's public player universe (auction values + true position eligibility). Everything is disk-cached; delete `cache/` to refresh.

In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — GAME LOGS (disk-cached)
# ═══════════════════════════════════════════════════════════════════
SEASONS       = ['2024-25', '2025-26']
CURRENT_SEASON, PREV_SEASON = '2025-26', '2024-25'
W_CURRENT, W_PREV = 0.70, 0.30      # recency weighting of the two seasons
MAX_GAMES     = 82

def fetch_gamelogs(seasons):
    frames = []
    for s in seasons:
        f = f'{CACHE_DIR}/gamelogs_{s}.csv'
        if os.path.exists(f):
            d = pd.read_csv(f)
            print(f'  📦 {s}: {len(d):,} rows from cache')
        else:
            print(f'  🌐 {s}: fetching from stats.nba.com ...', end='', flush=True)
            d = leaguegamelog.LeagueGameLog(
                season=s, player_or_team_abbreviation='P',
                season_type_all_star='Regular Season').get_data_frames()[0]
            d.to_csv(f, index=False)
            print(f' {len(d):,} rows -> cached')
            time.sleep(2.5)
        d['SEASON'] = s
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def parse_minutes(v):
    if isinstance(v, str) and ':' in v:
        a, b = v.split(':'); return int(a) + int(b) / 60
    try:    return float(v) if pd.notna(v) else 0.0
    except: return 0.0

df_logs = fetch_gamelogs(SEASONS)
df_logs['MIN']       = df_logs['MIN'].apply(parse_minutes)
df_logs['GAME_DATE'] = pd.to_datetime(df_logs['GAME_DATE'])
# A double-double = 10+ in two of PTS/REB/AST/STL/BLK, evaluated per game.
df_logs['DD'] = ((df_logs[['PTS','REB','AST','STL','BLK']] >= 10).sum(axis=1) >= 2).astype(int)

print(f"\n✅ {len(df_logs):,} player-games  ·  {df_logs['PLAYER_ID'].nunique():,} players")
print(df_logs.groupby('SEASON').agg(games=('GAME_ID','nunique'), rows=('PTS','size')).to_string())

  📦 2024-25: 26,306 rows from cache
  📦 2025-26: 26,651 rows from cache

✅ 52,957 player-games  ·  687 players
         games   rows
SEASON               
2024-25   1230  26306
2025-26   1230  26651


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — PLAYER METADATA: age · height · position (disk-cached)
# ═══════════════════════════════════════════════════════════════════
# Only players who appeared in 2025-26 are fetched (~400, not every player since 2015).
# Chunked with pauses and checkpointed after every chunk, so an interrupted run resumes
# instead of starting over.
#
# KNOWN LIMITATION: NBA.com exposes only coarse positions (Guard / Forward / Center and
# combos), never ESPN's PG vs SG vs SF vs PF. expand_position() below approximates ESPN
# eligibility from the coarse label plus height. It is a heuristic -- good enough to keep the
# roster optimiser honest about needing bigs vs guards, but CHECK ESPN for any player you are
# counting on to fill one specific starter slot.

META_FILE = f'{CACHE_DIR}/player_meta_{CURRENT_SEASON}.csv'
CHUNK, CHUNK_PAUSE, CALL_PAUSE = 25, 3.0, 0.6

need_ids = sorted(df_logs.loc[df_logs.SEASON == CURRENT_SEASON, 'PLAYER_ID'].unique())
have = pd.read_csv(META_FILE) if os.path.exists(META_FILE) else pd.DataFrame(columns=['PLAYER_ID'])
todo = [p for p in need_ids if p not in set(have['PLAYER_ID'])]
print(f'metadata: {len(have)} cached, {len(todo)} to fetch')

rows = []
for start in range(0, len(todo), CHUNK):
    for pid in todo[start:start+CHUNK]:
        for attempt in (1, 2):
            try:
                info = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0]
                h = info['HEIGHT'].iloc[0]
                ft_in = [int(x) for x in h.split('-')] if isinstance(h, str) and '-' in h else None
                rows.append({'PLAYER_ID': pid,
                             'BIRTH_DATE': info['BIRTHDATE'].iloc[0],
                             'POSITION':   info['POSITION'].iloc[0],
                             'HEIGHT_IN':  ft_in[0]*12 + ft_in[1] if ft_in else np.nan,
                             'DRAFT_YEAR': info['DRAFT_YEAR'].iloc[0],
                             'COUNTRY':    info['COUNTRY'].iloc[0]})
                time.sleep(CALL_PAUSE); break
            except Exception as e:
                if attempt == 2: print(f'  warn {pid}: {e}')
                else: time.sleep(2.0)
    if rows:
        have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True); rows = []
        have.to_csv(META_FILE, index=False)
        print(f'  checkpoint: {len(have)} players cached')
    time.sleep(CHUNK_PAUSE)

df_meta = have.drop_duplicates('PLAYER_ID').copy()

def expand_position(position, height_in):
    # Coarse NBA.com label + height -> the set of ESPN slots a player plausibly fills.
    p = (position or '').strip()
    h = height_in if pd.notna(height_in) else 78
    if   p == 'Guard':          s = {'PG','SG'} if h <= 76 else {'SG'}
    elif p == 'Guard-Forward':  s = {'SG','SF'} | ({'PG'} if h <= 76 else set())
    elif p == 'Forward-Guard':  s = {'SG','SF'} | ({'PF'} if h >= 80 else set())
    elif p == 'Forward':        s = {'SF','PF'} if h <= 82 else {'PF'}
    elif p == 'Forward-Center': s = {'PF','C'}  | ({'SF'} if h <= 82 else set())
    elif p == 'Center-Forward': s = {'PF','C'}
    elif p == 'Center':         s = {'C'}
    else:                       s = set()
    if not s:      # unknown label -> fall back to height alone
        s = ({'PG','SG'} if h <= 75 else {'SG','SF'} if h <= 78
             else {'SF','PF'} if h <= 81 else {'PF','C'})
    return s

df_meta['AGE'] = ((DRAFT_DATE - pd.to_datetime(df_meta['BIRTH_DATE'])).dt.days / 365.25).round(1)
df_meta['POS_ELIGIBLE'] = [expand_position(p, h)
                           for p, h in zip(df_meta['POSITION'], df_meta['HEIGHT_IN'])]
df_meta['POS_STR'] = [''.join(sorted(s, key=ALL_POSITIONS.index)) for s in df_meta['POS_ELIGIBLE']]

print(f'\n✅ metadata for {len(df_meta)} players')
print(df_meta[['PLAYER_ID','POSITION','HEIGHT_IN','AGE','POS_STR']].head(8).to_string(index=False))

metadata: 582 cached, 0 to fetch

✅ metadata for 582 players
 PLAYER_ID POSITION  HEIGHT_IN  AGE POS_STR
    203110  Forward         78 36.6    SFPF
   1627780    Guard         74 33.9    PGSG
    203500   Center         83 33.2       C
   1630578   Center         83 24.2       C
   1631106  Forward         80 25.4    SFPF
   1629216    Guard         74 30.3    PGSG
   1630559    Guard         77 28.4      SG
   1628983    Guard         78 28.3      SG


In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — CURRENT 2026-27 ROSTERS (disk-cached)
# ═══════════════════════════════════════════════════════════════════
# The game logs reflect each player's OLD team. The 2026 draft, trades and summer-2026 free
# agency reshuffled rosters, and usage is a function of who else is on the floor -- so Cell 6's
# teammate-strength adjustment needs to know where everyone plays NOW.

ROSTER_FILE = f'{CACHE_DIR}/team_rosters_2026-27.csv'
if os.path.exists(ROSTER_FILE):
    df_rosters = pd.read_csv(ROSTER_FILE)
    print(f'📦 rosters from cache: {len(df_rosters)} players, '
          f"{df_rosters['TEAM_ABBREVIATION'].nunique()} teams")
else:
    out = []
    for i, t in enumerate(static_teams.get_teams()):      # 30 static teams, no API call
        try:
            r = commonteamroster.CommonTeamRoster(
                    team_id=t['id'], season='2026-27').get_data_frames()[0]
            r['TEAM_ABBREVIATION'] = t['abbreviation']
            out.append(r[['PLAYER_ID','PLAYER','TEAM_ID','TEAM_ABBREVIATION']])
            time.sleep(0.8)
        except Exception as e:
            print(f"  warn {t['abbreviation']}: {e}")
        if (i+1) % 10 == 0 and out:
            pd.concat(out, ignore_index=True).to_csv(ROSTER_FILE, index=False)
            print(f'  checkpoint: {i+1}/30 teams'); time.sleep(3.0)
    df_rosters = pd.concat(out, ignore_index=True)
    df_rosters.to_csv(ROSTER_FILE, index=False)

print(f"✅ {len(df_rosters)} players on {df_rosters['TEAM_ABBREVIATION'].nunique()} rosters")

📦 rosters from cache: 621 players, 30 teams
✅ 621 players on 30 rosters


In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5B — ESPN MARKET DATA: auction values + true position eligibility
# ═══════════════════════════════════════════════════════════════════
# Public endpoint (no login). The private league endpoint
#   lm-api-reads.fantasy.espn.com/.../leagues/1015252784
# returns 401 without your espn_s2 / SWID cookies, so these are ESPN's
# league-agnostic values, rescaled below to this league's cap.
import json, urllib.request, unicodedata

ESPN_SEASON   = 2027
ESPN_CACHE    = f'{CACHE_DIR}/espn_market_{ESPN_SEASON}.json'
ESPN_URL      = (f'https://lm-api-reads.fantasy.espn.com/apis/v3/games/fba/'
                 f'seasons/{ESPN_SEASON}/players?view=kona_player_info')
ESPN_FILTER   = '{"players":{"limit":1500,"sortPercOwned":{"sortPriority":1,"sortAsc":false}}}'
USE_ESPN_MARKET    = True    # False -> fall back to this notebook's own VAR price model
USE_ESPN_POSITIONS = True    # False -> fall back to the Cell 4 height heuristic
# ESPN's raw values are calibrated for a smaller default league: for our pool the top 182 of them
# sum to about $1,832, while this league puts $2,800 on the table. Leaving them raw means only
# $1,832 of listed value is chasing $2,800 of budget, so the $200 cap stops binding -- the roster
# optimiser then buys a monstrous team for $200 and EVERY plan's win probability inflates
# (balanced measured 0.80 instead of ~0.50, which is nonsense). Prices in a market where money
# exceeds listed value clear ABOVE list, and this rescale is that inflation: it preserves ESPN's
# relative ordering, which is where the market information lives, and makes the budget bind.
#
# NOTE (v4): this uniform rescale (Jokic ~$128 in the final run) is only ONE of three market curves now.
# Cell 8 also builds a rank-based STRUCTURAL curve (Jokic ~$108) and a model-value curve, and the contested
# auctions are run under all three. The raw ESPN number stays alongside as ESPN_$_LIST, so you
# can see both on draft night. Set this False to bid strictly off ESPN's published numbers.
ESPN_SCALE_TO_CAP  = True

if os.path.exists(ESPN_CACHE):
    espn_raw = json.load(open(ESPN_CACHE))
    print(f'📦 ESPN market data from cache: {len(espn_raw)} players')
else:
    print('🌐 fetching ESPN player universe ...', end='', flush=True)
    req = urllib.request.Request(ESPN_URL, headers={
        'X-Fantasy-Filter': ESPN_FILTER,
        'Accept': 'application/json',
        'User-Agent': 'Mozilla/5.0',
    })
    try:                                   # python.org builds of Python on macOS have no root certificates of their own
        import ssl, certifi
        _ctx = ssl.create_default_context(cafile=certifi.where())
    except Exception:
        _ctx = None
    with urllib.request.urlopen(req, timeout=60, context=_ctx) as r:
        espn_raw = json.loads(r.read().decode())
    with open(ESPN_CACHE, 'w') as f:
        json.dump(espn_raw, f)
    print(f' {len(espn_raw)} players → cached')

# ESPN lineup-slot ids -> the five atomic positions this notebook reasons about.
# (5=G, 6=F, 7=SG/SF, 8=G/F, 9=PF/C, 10=F/C, 11=UTIL, 12=BENCH, 13=IR are all derived
#  from these five, so only 0-4 are needed.)
ESPN_SLOT = {0: 'PG', 1: 'SG', 2: 'SF', 3: 'PF', 4: 'C'}
ESPN_DEFAULT_POS = {1: 'PG', 2: 'SG', 3: 'SF', 4: 'PF', 5: 'C'}

SUFFIXES = (' jr', ' sr', ' ii', ' iii', ' iv', ' v')
def name_key(s):
    """'Nikola Jokić' -> 'nikolajokic'. ESPN strips accents and sometimes suffixes, so match on
    an accent-free, punctuation-free, suffix-free key."""
    s = ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))
    s = s.lower().replace('.', ' ').replace('-', ' ').replace("'", ' ')
    s = ' '.join(s.split())
    for suf in SUFFIXES:
        if s.endswith(suf):
            s = s[:-len(suf)]
            break
    return ''.join(ch for ch in s if ch.isalnum())

# ESPN's own per-game projection for 2027 (stat source 'projected', whole season: the numbers the draft room shows).
# Stat ids: 42 GP, 28 MIN, 29 PTS, 30 REB, 26 AST, 31 STL, 27 BLK, totals 17 3PM, 13 FGM, 14 FGA, 15 FTM, 16 FTA. ESPN has no
# double-double projection (it shows 0.0 for everybody), see the blend in Cell 6.
ESPN_PROJ_COLS = ['ESPN_GP', 'ESPN_MIN', 'ESPN_PTS', 'ESPN_REB', 'ESPN_AST', 'ESPN_STL', 'ESPN_BLK',
                  'ESPN_FG3M', 'ESPN_FGM', 'ESPN_FGA', 'ESPN_FTM', 'ESPN_FTA']
def espn_proj(p):
    for s in p.get('stats') or []:
        if s.get('id') == '102027' and s.get('stats'):
            st = s['stats']; gp = st.get('42') or 0
            if gp <= 0:
                break
            return {'ESPN_GP': gp, 'ESPN_MIN': st.get('28'), 'ESPN_PTS': st.get('29'), 'ESPN_REB': st.get('30'),
                    'ESPN_AST': st.get('26'), 'ESPN_STL': st.get('31'), 'ESPN_BLK': st.get('27'),
                    'ESPN_FG3M': st.get('17', 0) / gp, 'ESPN_FGM': st.get('13', 0) / gp, 'ESPN_FGA': st.get('14', 0) / gp,
                    'ESPN_FTM': st.get('15', 0) / gp, 'ESPN_FTA': st.get('16', 0) / gp}
    return {}

rows = []
for p in espn_raw:
    own = p.get('ownership') or {}
    slots = set(p.get('eligibleSlots') or ())
    pos = {ESPN_SLOT[i] for i in ESPN_SLOT if i in slots}
    rows.append({
        'KEY': name_key(p['fullName']),
        'ESPN_NAME': p['fullName'],
        'ESPN_$_RAW': own.get('auctionValueAverage'),
        'ESPN_PCT_OWNED': own.get('percentOwned'),
        'ESPN_POS_SET': pos if pos else None,
        'ESPN_PRIMARY': ESPN_DEFAULT_POS.get(p.get('defaultPositionId')),
        'ESPN_INJURY': p.get('injuryStatus'),
        **espn_proj(p),
        # ESPN's pre-draft rank for H2H category leagues (rank type STANDARD): the order of the draft-room player list
        'ESPN_RANK': ((p.get('draftRanksByRankType') or {}).get('STANDARD') or {}).get('rank'),
    })
df_espn = (pd.DataFrame(rows)
             .sort_values('ESPN_$_RAW', ascending=False, na_position='last')
             .drop_duplicates('KEY'))

for _c in ESPN_PROJ_COLS:
    if _c not in df_espn:
        df_espn[_c] = np.nan
print(f'   {int(df_espn["ESPN_GP"].notna().sum())} players carry an ESPN per-game projection for 2027')
n_val = df_espn['ESPN_$_RAW'].notna().sum()
top_sum = df_espn['ESPN_$_RAW'].nlargest(TOTAL_DRAFTED).sum()
print(f'   {n_val} players carry an ESPN auction value; '
      f'{df_espn["ESPN_POS_SET"].notna().sum()} carry eligible slots')
print(f'   ESPN top-{TOTAL_DRAFTED} values sum to ${top_sum:,.0f}; this league puts '
      f'${TOTAL_BUDGET:,} on the table')
print(f'   scale to cap: {ESPN_SCALE_TO_CAP}  ·  use ESPN market: {USE_ESPN_MARKET}'
      f'  ·  use ESPN positions: {USE_ESPN_POSITIONS}')
print('\n   ESPN market top 10:')
print(df_espn.head(10)[['ESPN_NAME','ESPN_$_RAW','ESPN_PRIMARY','ESPN_PCT_OWNED']]
        .to_string(index=False))

📦 ESPN market data from cache: 3220 players
   348 players carry an ESPN per-game projection for 2027
   1095 players carry an ESPN auction value; 2580 carry eligible slots
   ESPN top-182 values sum to $2,077; this league puts $2,800 on the table
   scale to cap: True  ·  use ESPN market: True  ·  use ESPN positions: True

   ESPN market top 10:
              ESPN_NAME  ESPN_$_RAW ESPN_PRIMARY  ESPN_PCT_OWNED
           Nikola Jokic       87.69            C           99.92
      Victor Wembanyama       83.50            C           99.93
            Luka Doncic       76.41           PG           99.90
Shai Gilgeous-Alexander       75.21           PG           99.92
  Giannis Antetokounmpo       65.99           PF           99.86
        Anthony Edwards       62.19           SG           99.90
        Cade Cunningham       58.78           PG           99.86
           Jayson Tatum       57.67           PF           99.84
          Jalen Johnson       55.88           PF           99.22
 

## Cell 6B · Availability — what past games played really tells you
v3 treated games played as a stable trait. This cell tests that on four seasons before anything is built on it.

In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6B — AVAILABILITY: how well do past games played predict next season's?
# ═══════════════════════════════════════════════════════════════════
# v3 set  AVAIL = mean(GP of the last two seasons) / 82  and then raised it to the power 1.35. That
# silently assumes games played is a stable player trait. It mostly is not, and this cell measures it
# instead of assuming it: four seasons of game logs, rotation players only (>=20 GP and >=18 MPG in the
# previous season, i.e. the people one actually drafts), predicting NEXT season's games played.
CAL_SEASONS = ['2022-23', '2023-24', '2024-25', '2025-26']
for s in CAL_SEASONS:
    f = f'{CACHE_DIR}/gamelogs_{s}.csv'
    if not os.path.exists(f):
        print(f'  fetching {s} ...', end='', flush=True)
        leaguegamelog.LeagueGameLog(season=s, player_or_team_abbreviation='P',
                                    season_type_all_star='Regular Season').get_data_frames()[0].to_csv(f, index=False)
        print(' cached'); time.sleep(3)
_gp, _mpg = {}, {}
for s in CAL_SEASONS:
    d = pd.read_csv(f'{CACHE_DIR}/gamelogs_{s}.csv'); d['m'] = d['MIN'].map(parse_minutes)
    _gp[s] = d.groupby('PLAYER_ID')['GAME_ID'].count(); _mpg[s] = d.groupby('PLAYER_ID')['m'].mean()
GPT = pd.DataFrame(_gp).fillna(0); MPT = pd.DataFrame(_mpg).fillna(0)
_bd = pd.to_datetime(df_meta.set_index('PLAYER_ID')['BIRTH_DATE'])
_rows = []
for t, l1, l2 in [('2024-25', '2023-24', '2022-23'), ('2025-26', '2024-25', '2023-24')]:
    for pid in GPT.index:
        if pid not in _bd.index or GPT.at[pid, l1] < 20 or MPT.at[pid, l1] < 18 or GPT.at[pid, t] < 1:
            continue
        age = (pd.Timestamp(f'{t[:4]}-10-01') - _bd[pid]).days / 365.25
        _rows.append(dict(y=GPT.at[pid, t]/82, g1=GPT.at[pid, l1]/82, g2=GPT.at[pid, l2]/82,
                          m1=MPT.at[pid, l1], age=age))
CAL = pd.DataFrame(_rows)
_X = np.c_[np.ones(len(CAL)), CAL.g1, CAL.g2, (CAL.m1-25)/10, np.maximum(CAL.age-30, 0)/5, (CAL.age < 23)*1.0]
AVAIL_COEF, *_ = np.linalg.lstsq(_X, CAL.y.to_numpy(), rcond=None)
_pred = _X @ AVAIL_COEF
_naive = (CAL.g1 + CAL.g2) / 2                                    # what v3 used
rm = lambda p: float(np.sqrt(((CAL.y - p)**2).mean()))
print(f'{len(CAL)} rotation-player seasons, mean next-season GP = {CAL.y.mean()*82:.1f} of 82')
print(f'\n  predictor of next-season games played                RMSE (lower = better)')
print(f'    same number for everybody (the mean)                 {rm(CAL.y.mean()):.3f}')
print(f'    v3: mean of last two seasons                         {rm(_naive):.3f}   <- WORSE than the constant')
print(f'    calibrated (this cell)                               {rm(_pred):.3f}')
print('\n  what actually happens to players, by games played LAST season:')
_b = pd.cut(CAL.g1*82, [0, 30, 50, 65, 75, 83], labels=['<30', '30-50', '50-65', '65-75', '75+'])
print(CAL.groupby(_b, observed=True).agg(n=('y','size'), last=('g1', lambda s: s.mean()*82),
                                        next_actual=('y', lambda s: s.mean()*82)).round(1).to_string())
print('\n  => games played persists only weakly. A 79-game player plays ~63 next year, not 79; a 42-game')
print('     player plays ~49, not 42. v3\'s spread (0.20-1.00) was about 2x too wide, and its convex')
print('     1.35 exponent then multiplied the error. Calibrated coefficients (const, g1, g2, mpg, age>30, age<23):')
print('    ', np.round(AVAIL_COEF, 3))

513 rotation-player seasons, mean next-season GP = 57.7 of 82

  predictor of next-season games played                RMSE (lower = better)
    same number for everybody (the mean)                 0.230
    v3: mean of last two seasons                         0.263   <- WORSE than the constant
    calibrated (this cell)                               0.219

  what actually happens to players, by games played LAST season:
         n  last  next_actual
g1                           
<30     24  25.4         48.0
30-50   73  42.0         49.2
50-65  133  58.9         55.4
65-75  169  70.9         60.8
75+    113  78.9         63.3

  => games played persists only weakly. A 79-game player plays ~63 next year, not 79; a 42-game
     player plays ~49, not 42. v3's spread (0.20-1.00) was about 2x too wide, and its convex
     1.35 exponent then multiplied the error. Calibrated coefficients (const, g1, g2, mpg, age>30, age<23):
     [ 0.392  0.331  0.067  0.028 -0.02   0.042]


## Cell 6 · Projections
Blend, calibrated availability, age curve, teammate context from the **change** in the supporting cast, and pool coverage for injured returnees. The per-game standard deviation is kept next to every mean: a category is won or lost on the distribution.

**New (2026-10-16): 50/50 blend with ESPN's own projection.** After the age and teammate adjustments every per-game mean is averaged with ESPN's 2027 per-game projection (the numbers its draft room shows), `ESPN_BLEND_W = 0.5`. ESPN projects no double-doubles (0.0 for everybody), so DD is derived from ESPN's stat line with a regression fitted on four seasons of game logs. Games played stays the model's own (calibrated availability plus the injury handling in the app).


In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — PROJECTIONS WITH PER-GAME VARIANCE   (v4)
# ═══════════════════════════════════════════════════════════════════
# Changes against v3, each backed by a measurement printed in the cells before/after:
#   * AVAILABILITY is the calibrated regression from Cell 6B, not the mean of two seasons.
#   * AGE curve keeps v3's shape (a walk-forward backtest confirms it) but 35+ is -9%, not -20%.
#   * TEAMMATE CONTEXT is driven by the CHANGE in the roster around a player, not its level. A star whose
#     supporting cast did not change already has that cast baked into last season's numbers; v3
#     applied the factor again (Jokic +12% for a Denver cast he already played with). Backtest: the change
#     explains 3.6x the variance of the level, and with both in the model the level coefficient is 0.
#   * POOL COVERAGE: players with no qualifying 2025-26 season (injured returnees, rookies) are on ESPN's
#     board and WILL be nominated, so they must exist in the pool for logging and inflation.
RAW = ['MIN','PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD']
GP_MIN, MPG_MIN = 15, 12
REQUIRE_ON_ROSTER = True      # a player on no NBA roster cannot be drafted
STAT_BETA = {'PTS': .07, 'FGM': .07, 'FGA': .07, 'FTM': .07, 'FTA': .07, 'AST': .06, 'FG3M': .05,
             'BLK': .05, 'STL': .04, 'DD': .04, 'REB': .03}      # per 1 sd of change in teammate strength

def season_agg(df):
    gb = df.groupby(['PLAYER_ID','PLAYER_NAME'])
    g = pd.concat([gb['GAME_ID'].count().rename('GP'), gb[RAW].mean().add_suffix('_M'),
                   gb[RAW].std().add_suffix('_S')], axis=1).reset_index()
    g['MPG'] = g['MIN_M']
    g = g[(g.GP >= GP_MIN) & (g.MPG >= MPG_MIN)].copy()
    for c in RAW: g[c+'_S'] = g[c+'_S'].fillna(0.0)
    return g

def age_factor(a):
    if pd.isna(a): return 1.0
    return (1.14 if a < 21 else 1.09 if a < 23 else 1.04 if a < 25 else 1.00 if a <= 28
            else 0.97 if a <= 30 else 0.93 if a <= 32 else 0.88 if a <= 34 else 0.91)

a_cur  = season_agg(df_logs[df_logs.SEASON == CURRENT_SEASON])
a_prev = season_agg(df_logs[df_logs.SEASON == PREV_SEASON])
P = a_cur.merge(a_prev, on=['PLAYER_ID','PLAYER_NAME'], how='left', suffixes=('','_P'))
P['ROW_TYPE'] = 'normal'

# ── pool coverage, part 1: rosterable LAST year, not this year (injured returnees) ──
cur_gp = df_logs[df_logs.SEASON == CURRENT_SEASON].groupby('PLAYER_ID')['GAME_ID'].count()
X = a_prev[~a_prev.PLAYER_ID.isin(a_cur.PLAYER_ID)].copy()
X['KEY'] = X['PLAYER_NAME'].map(name_key)
_on_roster = set(df_rosters['PLAYER'].map(name_key))
_espn_valued = set(df_espn.loc[df_espn['ESPN_$_RAW'].fillna(0) >= 1.0, 'KEY'])
X = X[X['KEY'].isin(_on_roster) & X['KEY'].isin(_espn_valued)].drop(columns='KEY')
if len(X):
    Xp = X.copy()
    for c in X.columns:
        if c not in ('PLAYER_ID', 'PLAYER_NAME'): Xp[c + '_P'] = X[c]          # blend = previous season only
    Xp['GP'] = Xp['PLAYER_ID'].map(cur_gp).fillna(0)                            # games he DID play this year
    Xp['ROW_TYPE'] = 'previous season only'
    P = pd.concat([P, Xp], ignore_index=True)

for c in RAW:
    cu, pv = P[c+'_M'], P[c+'_M_P']
    P['P_'+c+'_M'] = np.where(pv.notna(), W_CURRENT*cu + W_PREV*pv, cu)
    P['P_'+c+'_S'] = P[c+'_S']

P = P.merge(df_meta[['PLAYER_ID','AGE','HEIGHT_IN','POSITION','POS_ELIGIBLE','POS_STR']], on='PLAYER_ID', how='left')
P['AGE_FACTOR'] = P['AGE'].apply(age_factor).fillna(1.0)
P['HEIGHT_IN']  = P['HEIGHT_IN'].fillna(78)
P['POS_ELIGIBLE'] = [p if isinstance(p, set) else {'SG','SF'} for p in P['POS_ELIGIBLE']]

P['KEY'] = P['PLAYER_NAME'].map(name_key)
P = P.merge(df_espn[['KEY','ESPN_NAME','ESPN_$_RAW','ESPN_POS_SET','ESPN_PRIMARY','ESPN_INJURY','ESPN_PCT_OWNED','ESPN_RANK'] + ESPN_PROJ_COLS],
            on='KEY', how='left')
P['POS_SOURCE'] = 'height heuristic'
if USE_ESPN_POSITIONS:
    use_espn = P['ESPN_POS_SET'].notna()
    P.loc[use_espn, 'POS_SOURCE'] = 'ESPN'
    P['POS_ELIGIBLE'] = [set(b) if ok else a for ok, a, b in zip(use_espn, P['POS_ELIGIBLE'], P['ESPN_POS_SET'])]
P['POS_STR'] = [''.join(sorted(s, key=ALL_POSITIONS.index)) for s in P['POS_ELIGIBLE']]
print(f'📍 positions: {int((P.POS_SOURCE=="ESPN").sum())}/{len(P)} from ESPN · Center-eligible: '
      f'{sum(1 for s in P["POS_ELIGIBLE"] if "C" in s)}')

# ── availability: calibrated (Cell 6B). NOT raised to a power: production is linear in games played. ──
P['AVAIL'] = (AVAIL_COEF[0] + AVAIL_COEF[1]*P['GP']/MAX_GAMES + AVAIL_COEF[2]*P['GP_P'].fillna(0)/MAX_GAMES
              + AVAIL_COEF[3]*(P['MPG']-25)/10 + AVAIL_COEF[4]*np.maximum(P['AGE'].fillna(27)-30, 0)/5
              + AVAIL_COEF[5]*(P['AGE'].fillna(27) < 23)).clip(0.30, 0.95)
P['EXP_GP'] = (P['AVAIL'] * MAX_GAMES).round(0)
P['DUR_W']  = P['AVAIL'] ** DURABILITY_EXPONENT          # exponent is 1.0 in v4
P['RISK'] = np.select([P.AVAIL >= 0.76, P.AVAIL >= 0.68, P.AVAIL >= 0.60],
                      ['IRON MAN', 'STABLE', 'SOME RISK'], default='FRAGILE')

# ── current 2026-27 team (fall back to last 2025-26 team) ──
last_team = (df_logs[df_logs.SEASON == CURRENT_SEASON].sort_values('GAME_DATE')
             .groupby('PLAYER_ID')['TEAM_ABBREVIATION'].last().rename('TEAM_OLD').reset_index())
last_team_prev = (df_logs[df_logs.SEASON == PREV_SEASON].sort_values('GAME_DATE')
                  .groupby('PLAYER_ID')['TEAM_ABBREVIATION'].last().rename('TEAM_OLD_PREV').reset_index())
P = (P.merge(df_rosters[['PLAYER_ID','TEAM_ABBREVIATION']].rename(columns={'TEAM_ABBREVIATION':'TEAM'}),
             on='PLAYER_ID', how='left')
       .merge(last_team, on='PLAYER_ID', how='left').merge(last_team_prev, on='PLAYER_ID', how='left'))
# a player's ESPN/NBA ids differ and names are the only join key for rosters; fall back to the name key
_ros = df_rosters.assign(KEY=df_rosters['PLAYER'].map(name_key)).drop_duplicates('KEY').set_index('KEY')['TEAM_ABBREVIATION']
P['TEAM'] = P['TEAM'].fillna(P['KEY'].map(_ros))
P['ON_ROSTER_2627'] = P['TEAM'].notna()
P['TEAM_SOURCE'] = np.where(P['ON_ROSTER_2627'], '2026-27 roster', 'not rostered')
P['TEAM_OLD'] = P['TEAM_OLD'].fillna(P['TEAM_OLD_PREV'])
P['TEAM'] = P['TEAM'].fillna(P['TEAM_OLD']).fillna('FA')
if REQUIRE_ON_ROSTER:
    dropped = P.loc[~P['ON_ROSTER_2627'], ['PLAYER_NAME','TEAM_OLD','P_PTS_M']]
    P = P[P['ON_ROSTER_2627']].copy()
    print(f'🚫 removed {len(dropped)} players not on any 2026-27 roster (retired / unsigned): '
          + ', '.join(dropped.sort_values('P_PTS_M', ascending=False).PLAYER_NAME.head(8)) + ' ...')

for c in ['PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD','MIN']:
    P['P_'+c+'_M'] *= P['AGE_FACTOR']                   # age moves production, not variance

# ── teammate context from the CHANGE in the supporting cast ──────────────────────────────────────
# star score = z-sum of last season's per-game production (fallback: the season before), computed on ONE
# scale for everybody, so "who played with him last year" and "who plays with him now" are comparable.
_latest = pd.concat([a_cur, a_prev[~a_prev.PLAYER_ID.isin(a_cur.PLAYER_ID)]], ignore_index=True)
_star = pd.Series(0.0, index=_latest.index)
for c in ['PTS','REB','AST','STL','BLK','FG3M']:
    v = _latest[c+'_M']; _star += (v - v.mean()) / v.std()
STAR = dict(zip(_latest['PLAYER_ID'], _star))
_old_team = (df_logs[df_logs.SEASON == CURRENT_SEASON].sort_values('GAME_DATE')
             .groupby('PLAYER_ID')['TEAM_ABBREVIATION'].last())
_old_sum = pd.Series({pid: STAR.get(pid, 0.0) for pid in a_cur.PLAYER_ID}).groupby(_old_team.reindex(a_cur.PLAYER_ID).values).sum()
_new_roster = df_rosters.drop_duplicates('PLAYER_ID')
_new_sum = pd.Series({r.PLAYER_ID: STAR.get(r.PLAYER_ID, 0.0) for r in _new_roster.itertuples()}) \
             .groupby(_new_roster.set_index('PLAYER_ID')['TEAM_ABBREVIATION'].values).sum()
P['STAR_SCORE'] = P['PLAYER_ID'].map(STAR).fillna(0.0)
P['MATES_NEW'] = P['TEAM'].map(_new_sum).fillna(0.0) - np.where(P['PLAYER_ID'].isin(_new_roster['PLAYER_ID']), P['STAR_SCORE'], 0.0)
P['MATES_OLD'] = P['TEAM_OLD'].map(_old_sum).fillna(0.0) - np.where(P['PLAYER_ID'].isin(a_cur['PLAYER_ID']), P['STAR_SCORE'], 0.0)
_no_ctx = P['TEAM_OLD'].isna() | (P['ROW_TYPE'] != 'normal')      # no 2025-26 baseline to compare against
P.loc[_no_ctx, 'MATES_OLD'] = P.loc[_no_ctx, 'MATES_NEW']
_sd = max(float(P['MATES_NEW'].std()), 1e-9)
P['MATES_DELTA'] = (P['MATES_NEW'] - P['MATES_OLD']) / _sd         # in sd units; 0 = same supporting cast as last year
for c in ['PTS','REB','AST','STL','BLK','FG3M','FGM','FGA','FTM','FTA','DD']:
    P['P_'+c+'_M'] *= (1 - STAT_BETA[c] * P['MATES_DELTA']).clip(0.85, 1.15)
P['CTX'] = (1 - STAT_BETA['PTS'] * P['MATES_DELTA']).clip(0.85, 1.15)
P['MATES'] = P['MATES_NEW']

# ── blend with ESPN's own projection ─────────────────────────────────────────────────────────────
# ESPN publishes a per-game projection for 2027 (the numbers its draft room shows). It is an independent forecast, so
# averaging it with ours is usually better than either alone. Availability (games played) stays OUR number: ESPN's GP
# projection is optimistic (Jokic 72) and knows nothing about the injury handling.
# DOUBLE-DOUBLES: ESPN does not project them (it shows 0.0 for everybody), so they are DERIVED from ESPN's own stat line:
# P(double-double) as a function of the five per-game means (sorted: PTS/REB/AST/STL/BLK, a logistic fitted on four
# seasons of game logs; leave-one-season-out RMSE about 0.03 against a spread of 0.155 between players).
ESPN_BLEND_W = 0.5            # share of ESPN in every per-game mean: 0 = model only, 1 = ESPN only

def _dd_features(df):
    X5 = np.sort(df[['PTS', 'REB', 'AST', 'STL', 'BLK']].to_numpy(float) / 10.0, axis=1)[:, ::-1]
    s1, s2, s3 = X5[:, 0], X5[:, 1], X5[:, 2]
    return np.c_[np.ones(len(df)), s1, s2, s3, s2 ** 2, s1 * s2, s2 ** 3]

def _fit_dd_logit(X, y, w, iters=60, l2=1e-3):
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(X @ b))); W = w * p * (1 - p) + 1e-9
        step = np.linalg.solve((X * W[:, None]).T @ X + l2 * np.eye(X.shape[1]), X.T @ (w * (y - p)) - l2 * b)
        b += step
        if np.abs(step).max() < 1e-8:
            break
    return b

_dd_frames = []
for _s in ['2022-23', '2023-24', '2024-25', '2025-26']:
    _f = f'{CACHE_DIR}/gamelogs_{_s}.csv'
    if os.path.exists(_f):
        _dd_frames.append(pd.read_csv(_f, usecols=['PLAYER_ID', 'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK']).assign(SEASON=_s))
_dd_logs = pd.concat(_dd_frames) if _dd_frames else df_logs[['PLAYER_ID', 'SEASON', 'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK']]
_dd_logs = _dd_logs.assign(DDF=((_dd_logs[['PTS', 'REB', 'AST', 'STL', 'BLK']] >= 10).sum(axis=1) >= 2).astype(int))
_dd_ps = _dd_logs.groupby(['SEASON', 'PLAYER_ID']).agg(GP=('DDF', 'size'), MIN=('MIN', 'mean'), PTS=('PTS', 'mean'), REB=('REB', 'mean'),
                                                       AST=('AST', 'mean'), STL=('STL', 'mean'), BLK=('BLK', 'mean'), DD=('DDF', 'mean')).reset_index()
_dd_ps = _dd_ps[(_dd_ps.GP >= 15) & (_dd_ps.MIN >= 12)]
DD_COEF = _fit_dd_logit(_dd_features(_dd_ps), _dd_ps.DD.to_numpy(float), _dd_ps.GP.to_numpy(float))
_dd_pred = lambda df: 1 / (1 + np.exp(-(_dd_features(df) @ DD_COEF)))
_in = _dd_ps.DD.to_numpy(float); _pr = _dd_pred(_dd_ps)
print(f'🔀 DD from a stat line: fitted on {len(_dd_ps)} player-seasons, in-sample RMSE {np.sqrt(np.average((_pr - _in) ** 2, weights=_dd_ps.GP)):.3f}')

_has = P['ESPN_GP'].notna() & (P['ESPN_GP'] > 0)
_espn_dd = pd.Series(_dd_pred(P[['ESPN_PTS', 'ESPN_REB', 'ESPN_AST', 'ESPN_STL', 'ESPN_BLK']].fillna(0.0).rename(columns=lambda c: c[5:])), index=P.index)
P['ESPN_DD_IMPLIED'] = np.where(_has, _espn_dd, np.nan)
for c in ['PTS', 'REB', 'AST', 'DD']:
    P['MODEL_' + c] = P['P_' + c + '_M']                                       # kept for the comparison below and the export
_before = {c: P['P_' + c + '_M'].copy() for c in ['MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'FGM', 'FGA', 'FTM', 'FTA', 'DD']}
for c in ['MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'FGM', 'FGA', 'FTM', 'FTA', 'DD']:
    e = P['ESPN_DD_IMPLIED'] if c == 'DD' else P['ESPN_' + c]
    P['P_' + c + '_M'] = np.where(_has & e.notna(), (1 - ESPN_BLEND_W) * P['P_' + c + '_M'] + ESPN_BLEND_W * e, P['P_' + c + '_M'])
print(f'🔀 blended {int(_has.sum())} of {len(P)} players with ESPN\'s projection at {ESPN_BLEND_W:.0%} ESPN / {1 - ESPN_BLEND_W:.0%} model')
print('   mean change per game among blended players: ' + ', '.join(
    f'{c} {(P.loc[_has, "P_" + c + "_M"] - _before[c][_has]).mean():+.2f}' for c in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'DD']))
_cmp = P.loc[_has].assign(GAP=lambda d: d['ESPN_PTS'] - d['MODEL_PTS'])
print('   where ESPN and the model disagree most on points per game (ESPN - model):')
print(pd.concat([_cmp.nsmallest(4, 'GAP'), _cmp.nlargest(6, 'GAP')])[['PLAYER_NAME', 'MODEL_PTS', 'ESPN_PTS', 'MODEL_DD', 'ESPN_DD_IMPLIED']].round(2).to_string(index=False))

P = P.reset_index(drop=True)
print(f'✅ {len(P)} players projected:  {P.ROW_TYPE.value_counts().to_dict()}')
print(f'   availability: mean {P.AVAIL.mean():.2f}, sd {P.AVAIL.std():.2f}, range {P.AVAIL.min():.2f}-{P.AVAIL.max():.2f}'
      f'   (v3: mean 0.71, sd 0.17, range 0.20-1.00)')
print(f'   teammate-change factor (PTS) range {P.CTX.min():.2f}-{P.CTX.max():.2f}; '
      f'{int((P.MATES_DELTA.abs() < 0.05).sum())} players have an unchanged supporting cast')
print('\nInjured returnees: no qualifying 2025-26 season, so projected from 2024-25 with a low calibrated availability:')
print(P[P.ROW_TYPE != 'normal'].sort_values('ESPN_$_RAW', ascending=False)
        [['PLAYER_NAME', 'TEAM', 'AGE', 'AVAIL', 'ESPN_$_RAW', 'ESPN_INJURY']].round(2).to_string(index=False))

📍 positions: 401/404 from ESPN · Center-eligible: 86
🚫 removed 28 players not on any 2026-27 roster (retired / unsigned): Cam Thomas, Russell Westbrook, Jaden Ivey, D'Angelo Russell, Jonas Valanciunas, Sharife Cooper, Tyson Etienne, Cole Anthony ...


🔀 DD from a stat line: fitted on 1545 player-seasons, in-sample RMSE 0.030
🔀 blended 303 of 376 players with ESPN's projection at 50% ESPN / 50% model
   mean change per game among blended players: PTS +0.26, REB +0.08, AST +0.06, STL +0.01, BLK +0.02, FG3M +0.02, DD +0.01
   where ESPN and the model disagree most on points per game (ESPN - model):
          PLAYER_NAME  MODEL_PTS  ESPN_PTS  MODEL_DD  ESPN_DD_IMPLIED
         Tyrese Maxey      28.16     24.85      0.12             0.09
       Damian Lillard      24.95     21.65      0.24             0.13
       Keyonte George      25.33     22.27      0.08             0.12
       Isaiah Collier      12.64      9.85      0.19             0.11
Giannis Antetokounmpo      23.83     32.06      0.54             0.79
        Pascal Siakam      17.08     22.80      0.12             0.23
           Trae Young      18.17     23.82      0.32             0.62
   Michael Porter Jr.      18.46     23.50      0.19             0.21
        Anthony Dav

## Cell 7 · Category structure
Correlation and concentration, unchanged from v3. Read these as description, not as a plan: the strongly negative pairs (FG% vs 3PM −48, FG% vs FT% −40) are why a roster cannot own everything, and the optimiser in Cell 11 deals with them by construction because it builds actual rosters, not category wish-lists.

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — CATEGORY STRUCTURE: correlation + concentration
# ═══════════════════════════════════════════════════════════════════
# Per-player category rates, as a manager would read them off a projection page.
def player_cat_rates(P):
    d = pd.DataFrame({'PLAYER_ID': P['PLAYER_ID'], 'PLAYER_NAME': P['PLAYER_NAME']})
    for c in COUNTING_CATS:
        d[c] = P['P_'+c+'_M']
    d['FG_PCT']  = np.where(P['P_FGA_M'] > 0, P['P_FGM_M'] / P['P_FGA_M'], np.nan)
    d['FT_PCT']  = np.where(P['P_FTA_M'] > 0, P['P_FTM_M'] / P['P_FTA_M'], np.nan)
    d['PTS_MIN'] = np.where(P['P_MIN_M'] > 0, P['P_PTS_M'] / P['P_MIN_M'], np.nan)
    # volumes: needed because a ratio only matters in proportion to attempts
    d['FGA'] = P['P_FGA_M']; d['FTA'] = P['P_FTA_M']; d['MIN'] = P['P_MIN_M']
    return d

rates = player_cat_rates(P)

# ── 1. correlation between categories, across the rosterable pool ──
Zraw = rates[CATS].apply(lambda s: (s - s.mean()) / s.std())
corr = Zraw.corr()
corr.index = [LBL[c] for c in corr.index]; corr.columns = [LBL[c] for c in corr.columns]
print('CATEGORY CORRELATION  (x100, across the rosterable pool)')
print('   read for CLUSTERS: categories that move together are cheap to buy together')
print((corr*100).round(0).astype(int).to_string())

# ── 2. concentration: how top-heavy is each category's supply? ──
# Measured in the units that actually move a TEAM total (Cell 8's "impact"), so counting and
# ratio categories are directly comparable: EDGE = how many standard deviations of category
# advantage the elite players hand you. High EDGE_TOP3 means a couple of buys win the
# category outright -- which is exactly what makes a category cheap to dominate.
def gini(x):
    x = np.sort(np.clip(np.asarray(x, float), 0, None))
    n, s = len(x), x.sum()
    return float(((2*np.arange(1, n+1) - n - 1) * x).sum() / (n*s)) if s > 0 else 0.0

def impact_units(rates):
    out = {}
    for c in COUNTING_CATS:
        out[c] = rates[c].to_numpy(float)
    for c, vol in [('FG_PCT','FGA'), ('FT_PCT','FTA'), ('PTS_MIN','MIN')]:
        lg = (rates[vol] * rates[c]).sum() / rates[vol].sum()
        out[c] = ((rates[c] - lg) * rates[vol]).to_numpy(float)
    return out

_imp = impact_units(rates)
conc = []
for c in CATS:
    v = _imp[c]; v = v[~np.isnan(v)]
    z = (v - v.mean()) / v.std()
    top = np.sort(z)[::-1]
    conc.append({'CAT': LBL[c],
                 'EDGE_TOP3':  top[:3].mean(),
                 'EDGE_TOP20': top[:20].mean(),
                 'GINI': gini(v)})
df_conc = pd.DataFrame(conc).sort_values('EDGE_TOP3', ascending=False)

print('\n\nCATEGORY CONCENTRATION  —  how much edge the elite hand you, in pool sigmas')
print('   EDGE_TOP3/20 = mean z-score of the best 3 / best 20 assets in that category')
print('   higher = fewer buys needed to own the category = CHEAP to dominate')
print(df_conc.round(2).to_string(index=False))
print(f"\n   Cheapest to dominate : {' > '.join(df_conc.head(3).CAT)}"
      f"  <- a few buys can win these outright")
print(f"   Most expensive       : {' > '.join(df_conc.tail(3).CAT)}"
      f"  <- edge is spread thin, you must outspend the field")

CATEGORY CORRELATION  (x100, across the rosterable pool)
   read for CLUSTERS: categories that move together are cheap to buy together
     FG%  FT%  3PM  REB  AST  STL  BLK   DD  PTS  PPM
FG%  100  -41  -48   55  -11  -12   47   37    3    5
FT%  -41  100   61  -26   33   13  -28   -9   40   42
3PM  -48   61  100  -14   43   27  -20   -3   61   56
REB   55  -26  -14  100   20   17   65   83   43   32
AST  -11   33   43   20  100   54   -6   47   70   58
STL  -12   13   27   17   54  100    6   21   45   29
BLK   47  -28  -20   65   -6    6  100   48   18   11
DD    37   -9   -3   83   47   21   48  100   51   42
PTS    3   40   61   43   70   45   18   51  100   92
PPM    5   42   56   32   58   29   11   42   92  100


CATEGORY CONCENTRATION  —  how much edge the elite hand you, in pool sigmas
   EDGE_TOP3/20 = mean z-score of the best 3 / best 20 assets in that category
   higher = fewer buys needed to own the category = CHEAP to dominate
CAT  EDGE_TOP3  EDGE_TOP20  GINI
BLK       5

## Cell 8 · Valuation and the three market curves
Volume-weighted z-scores (a 92% shooter on 1 FTA/g moves team FT% by almost nothing) and three explicit market curves. The notebook measured that the best category plan changes with the curve, so the draft assistant treats the price *level* as something it learns during the auction (money conservation: all remaining budgets get spent on the players still to come) rather than something it knows.

In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — VALUATION: volume-weighted z-scores, three market curves
# ═══════════════════════════════════════════════════════════════════
def category_impact(rates):
    # Per-category contribution in the units that actually move a TEAM total.
    # Counting cats: per-game production. Ratio cats: (rate - league rate) * volume.
    imp = pd.DataFrame(index=rates.index)
    for c in COUNTING_CATS:
        imp[c] = rates[c]
    lg = {}
    for c, vol in [('FG_PCT','FGA'), ('FT_PCT','FTA'), ('PTS_MIN','MIN')]:
        lg[c] = (rates[vol] * rates[c]).sum() / rates[vol].sum()
        imp[c] = (rates[c] - lg[c]) * rates[vol]
    return imp, lg

impact, LEAGUE_RATE = category_impact(rates)
Z = pd.DataFrame({'Z_'+c: (impact[c] - impact[c].mean()) / impact[c].std() for c in CATS})
P = pd.concat([P, Z], axis=1)

def strategy_value(Z, targets, dur_w):
    # Value under a punt plan: only the target categories count, scaled by the convex
    # durability weight -- an absent player posts zeros in a locked weekly lineup.
    return sum(Z['Z_'+c] for c in targets) * dur_w

def auction_prices(value, n_drafted=TOTAL_DRAFTED, budget=TOTAL_BUDGET, min_bid=MIN_BID):
    # Value above replacement -> dollars. Replacement = the n_drafted-th best player.
    v = np.asarray(value, float)
    order = np.argsort(-v)
    repl = v[order[min(n_drafted, len(v)) - 1]]
    var = np.clip(v - repl, 0, None)
    pot = budget - n_drafted*min_bid
    tot = var[order[:n_drafted]].sum()
    if tot <= 0:
        return np.full(len(v), min_bid, int)
    drafted = np.zeros(len(v), bool); drafted[order[:n_drafted]] = True
    return np.where(drafted, np.round((var/tot)*pot + min_bid), 0).astype(int)

# ── MARKET price: what the room will actually pay ──
# Three explicit market models. The true price curve of THIS league is unknowable before the draft, and the
# notebook measured that the best category plan CHANGES with it, so none of them is treated as truth:
#   UNIFORM     ESPN's published averages stretched uniformly until the top 182 sum to the $2,800 pot
#               (v3's assumption; it makes stars look ~58% dearer than ESPN lists them)
#   STRUCTURAL  ESPN's averages behave like a ~10-team curve (their top 130 sum to $1,833 of a $2,000 pot, and ranks
#               131+ are ~$0). 14 teams turn ranks 100-182 into STARTERS, so the mid-tier is lifted a lot and
#               stars only 10-20%.  Jokic: $85 ESPN list -> ~$127 uniform, ~$113 structural.
#   MODEL       the value-above-replacement price of this notebook's own balanced valuation (no ESPN information)
MARKET_MODEL = 'STRUCTURAL'          # which curve MARKET_$ carries; the others stay as columns
ESPN_LEAGUE_TEAMS = 10
P['VAL_MARKET'] = strategy_value(Z, CATS, P['DUR_W'])
P['MODEL_$']    = auction_prices(P['VAL_MARKET'])
import auction_sim as A
_espn = P['ESPN_$_RAW'].astype(float).to_numpy()
_mk = {'UNIFORM': A.market_uniform(_espn, TOTAL_DRAFTED, TOTAL_BUDGET, MIN_BID),
       'STRUCTURAL': A.market_structural(_espn, P['VAL_MARKET'].to_numpy(), NUM_TEAMS, ESPN_LEAGUE_TEAMS,
                                         BUDGET_PER_TEAM, ROSTER_SIZE, MIN_BID)}
for _k, _v in _mk.items():
    P['MARKET_' + _k] = np.maximum(np.where(np.isnan(_v), P['MODEL_$'], np.round(_v)), 0).astype(int)
P['MARKET_MODEL'] = P['MODEL_$']
P['ESPN_$_LIST'] = P['ESPN_$_RAW'].round()                       # what ESPN's own UI shows you
P['MARKET_$'] = P['MARKET_' + MARKET_MODEL]
P['MARKET_SOURCE'] = np.where(np.isnan(_espn), 'model', 'ESPN')
print(f'💰 MARKET_$ = {MARKET_MODEL} curve.  Top-182 sums: ' + ' · '.join(
      f'{k} ${P["MARKET_"+k].nlargest(TOTAL_DRAFTED).sum():,}' for k in ['UNIFORM', 'STRUCTURAL', 'MODEL']))
print('   Top 5 under each curve   (ESPN list → UNIFORM / STRUCTURAL / MODEL)')
for _, r in P.nlargest(5, 'ESPN_$_RAW').iterrows():
    print(f"     {r.PLAYER_NAME:<26} ${r['ESPN_$_LIST']:>4.0f} → ${r.MARKET_UNIFORM:>4} / ${r.MARKET_STRUCTURAL:>4} / ${r.MARKET_MODEL:>4}")

_top = P['MARKET_$'].nlargest(TOTAL_DRAFTED).sum()
print(f"✅ market board priced. Top {TOTAL_DRAFTED} (the players who get drafted) = ${_top:,}"
      f" against a ${TOTAL_BUDGET:,} league pot.")
print(f"   (a few dollars either side of the pot is rounding; the contested-auction cell spends real budgets.)")
print(f"   league rates: FG% {LEAGUE_RATE['FG_PCT']:.3f} · FT% {LEAGUE_RATE['FT_PCT']:.3f} "
      f"· PPM {LEAGUE_RATE['PTS_MIN']:.3f}")
print('\nTOP 15 — MARKET BOARD (what the room will pay)')
print(P.nlargest(15,'MARKET_$')[['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK',
                                 'MARKET_$','P_PTS_M','P_REB_M','P_AST_M']]
       .round(2).to_string(index=False))

# ── where ESPN and the model disagree: the market's soft spots ──
if (P['MARKET_SOURCE'] == 'ESPN').any():
    cmpm = P[P['MARKET_SOURCE'] == 'ESPN'].copy()
    cmpm['GAP'] = cmpm['MARKET_$'] - cmpm['MODEL_$']
    CG = ['PLAYER_NAME','TEAM','POS_STR','AGE','EXP_GP','RISK','MARKET_$','MODEL_$','GAP']
    print('\n\nESPN MARKET vs THIS MODEL — the market pays UP for these (fade them):')
    print(cmpm.nlargest(10, 'GAP')[CG].to_string(index=False))
    print('\n   ...and the market DISCOUNTS these relative to the model (look here first):')
    print(cmpm.nsmallest(10, 'GAP')[CG].to_string(index=False))
    print('\n   Two causes, and the first one is the whole opportunity:')
    print('   1. CATEGORY MISMATCH. ESPN prices players for its default 9-cat scoring, which has'
          '\n      no DD and no PPM. This league scores both. So rebounding centres -- who pile up'
          '\n      REB, DD and FG% together -- are priced as if double-doubles were worth nothing'
          '\n      to anyone. That is a structural discount the room inherits from ESPN, and it is'
          '\n      exactly where a REB/DD plan shops.')
    print('   2. AVAILABILITY. v3 blamed fragile stars (a convex games-played penalty, exponent 1.35) for much of this'
          '\n      gap. Cell 6B measured that games played barely persists year to year, so v4 uses the calibrated,'
          '\n      much flatter availability and a LINEAR weight. What gap remains is real ESPN-vs-model disagreement.')

# Demonstration of the volume-weighting fix on FT%
naive = (rates['FT_PCT'] - rates['FT_PCT'].mean()) / rates['FT_PCT'].std()
cmp = pd.DataFrame({'PLAYER': rates['PLAYER_NAME'], 'FT%': rates['FT_PCT'],
                    'FTA/g': rates['FTA'], 'NAIVE_Z': naive, 'VOLUME_WEIGHTED_Z': Z['Z_FT_PCT']})
print('\n\nWHY VOLUME-WEIGHTING MATTERS — top 8 by the NAIVE (raw-percentage) FT% z-score:')
print(cmp.nlargest(8,'NAIVE_Z').round(2).to_string(index=False))
print('\n   ...and the top 8 by volume-weighted FT% impact (who actually moves your team FT%):')
print(cmp.nlargest(8,'VOLUME_WEIGHTED_Z').round(2).to_string(index=False))

💰 MARKET_$ = STRUCTURAL curve.  Top-182 sums: UNIFORM $2,864 · STRUCTURAL $2,805 · MODEL $2,801
   Top 5 under each curve   (ESPN list → UNIFORM / STRUCTURAL / MODEL)
     Nikola Jokić               $  88 → $ 123 / $ 105 / $  70
     Victor Wembanyama          $  84 → $ 117 / $ 101 / $  73
     Luka Dončić                $  76 → $ 107 / $  94 / $  56
     Shai Gilgeous-Alexander    $  75 → $ 105 / $  93 / $  55
     Giannis Antetokounmpo      $  66 → $  92 / $  83 / $  32
✅ market board priced. Top 182 (the players who get drafted) = $2,805 against a $2,800 league pot.
   (a few dollars either side of the pot is rounding; the contested-auction cell spends real budgets.)
   league rates: FG% 0.473 · FT% 0.785 · PPM 0.486

TOP 15 — MARKET BOARD (what the room will pay)
            PLAYER_NAME TEAM POS_STR  AGE  EXP_GP     RISK  MARKET_$  P_PTS_M  P_REB_M  P_AST_M
           Nikola Jokić  DEN       C 31.7    60.0   STABLE       105    26.88    12.24     9.85
      Victor Wembanyama  SAS  

## Cell 8A · Pool coverage, part 2 — ESPN-priced players with no stats
Rookies have no game logs but are on ESPN's board. They get a value-consistent proxy line, are loggable, and are never recommended.

In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8A — POOL COVERAGE, PART 2: players ESPN prices that have no stats at all
# ═══════════════════════════════════════════════════════════════════
# 2026-draft rookies have never played an NBA game, so no projection exists, but ESPN prices them and the
# room WILL bid on them. If the draft assistant cannot log a sale it cannot track the money that left the
# room, and the inflation arithmetic goes wrong for the rest of the night.
# So each one gets a PROXY line: the player whose MODEL value is closest to the price this player will fetch
# (value-consistent by construction, so a proxy is never a phantom bargain), flagged MODEL_OK = False. The
# optimiser never recommends a MODEL_OK = False player; they exist for logging and inflation only.
P['MODEL_OK'] = True; P['PROXY_PID'] = np.nan
_espn_id = {name_key(p['fullName']): p['id'] for p in espn_raw}
_ros_team = df_rosters.assign(KEY=df_rosters['PLAYER'].map(name_key)).drop_duplicates('KEY').set_index('KEY')['TEAM_ABBREVIATION']
_missing = df_espn[(df_espn['ESPN_$_RAW'].fillna(0) >= 1.0) & ~df_espn['KEY'].isin(set(P['KEY']))
                   & df_espn['KEY'].isin(set(_ros_team.index))]
_srt = P.dropna(subset=['ESPN_$_RAW']).sort_values('ESPN_$_RAW')
def _curve(col, x):                       # ESPN list price -> price under a market curve (monotone interpolation)
    return float(np.interp(x, _srt['ESPN_$_RAW'].to_numpy(), np.maximum.accumulate(_srt[col].to_numpy(float))))
_base = P.copy(); _syn = []
for r in _missing.to_dict('records'):
    mk = {k: _curve('MARKET_' + k, r['ESPN_$_RAW']) for k in ['UNIFORM', 'STRUCTURAL']}
    j = (_base['MODEL_$'] - mk[MARKET_MODEL]).abs().idxmin()
    row = _base.loc[j].copy()
    row['PROXY_PID'] = _base.at[j, 'PLAYER_ID']
    row['PLAYER_ID'], row['PLAYER_NAME'], row['KEY'] = int(_espn_id[r['KEY']]), r['ESPN_NAME'], r['KEY']
    row['ESPN_$_RAW'], row['ESPN_$_LIST'], row['ESPN_INJURY'] = r['ESPN_$_RAW'], round(r['ESPN_$_RAW']), r['ESPN_INJURY']
    row['ESPN_RANK'] = r['ESPN_RANK']
    row['MARKET_UNIFORM'], row['MARKET_STRUCTURAL'] = int(round(mk['UNIFORM'])), int(round(mk['STRUCTURAL']))
    row['MARKET_MODEL'] = row['MODEL_$']
    row['MARKET_$'] = row['MARKET_' + MARKET_MODEL]
    row['MARKET_SOURCE'], row['MODEL_OK'], row['ROW_TYPE'] = 'ESPN', False, 'ESPN-only (proxy line)'
    row['TEAM'], row['TEAM_SOURCE'], row['ON_ROSTER_2627'] = _ros_team.get(r['KEY']), '2026-27 roster', True
    row['AGE'], row['AVAIL'], row['RISK'] = 22.0, 0.62, 'ROOKIE/UNKNOWN'
    row['EXP_GP'], row['DUR_W'] = round(0.62 * MAX_GAMES), 0.62
    if isinstance(r['ESPN_POS_SET'], set):
        row['POS_ELIGIBLE'], row['POS_SOURCE'] = set(r['ESPN_POS_SET']), 'ESPN'
    row['POS_STR'] = ''.join(sorted(row['POS_ELIGIBLE'], key=ALL_POSITIONS.index))
    _syn.append(row)
if _syn:
    P = pd.concat([P, pd.DataFrame(_syn)], ignore_index=True)
P = P.reset_index(drop=True)
print(f'✅ pool: {len(P)} players  {P.ROW_TYPE.value_counts().to_dict()}')
print(f'   availability: mean {P.AVAIL.mean():.2f}, sd {P.AVAIL.std():.2f}, range {P.AVAIL.min():.2f}-{P.AVAIL.max():.2f}'
      f'   (v3: mean 0.71, sd 0.17, range 0.20-1.00)')
print('\nPlayers v3 could not log (a nominated one would have broken the draft log and the inflation arithmetic):')
print(P[P.ROW_TYPE != 'normal'].sort_values('ESPN_$_RAW', ascending=False)
        [['PLAYER_NAME', 'TEAM', 'ROW_TYPE', 'AVAIL', 'ESPN_$_LIST', 'MARKET_$', 'MODEL_$', 'ESPN_INJURY']]
        .head(14).round(2).to_string(index=False))

✅ pool: 382 players  {'normal': 369, 'previous season only': 7, 'ESPN-only (proxy line)': 6}
   availability: mean 0.67, sd 0.09, range 0.42-0.84   (v3: mean 0.71, sd 0.17, range 0.20-1.00)

Players v3 could not log (a nominated one would have broken the draft log and the inflation arithmetic):
      PLAYER_NAME TEAM               ROW_TYPE  AVAIL  ESPN_$_LIST  MARKET_$  MODEL_$ ESPN_INJURY
Tyrese Haliburton  IND   previous season only   0.48         41.0        54       25      ACTIVE
     Kyrie Irving  DAL   previous season only   0.46         12.0        17       21  DAY_TO_DAY
   Walker Kessler  LAL   previous season only   0.47         10.0        15       16      ACTIVE
  Dejounte Murray  NOP   previous season only   0.49          8.0        12       16  DAY_TO_DAY
   Cameron Boozer  MEM ESPN-only (proxy line)   0.62          7.0        12       12      ACTIVE
   Damian Lillard  POR   previous season only   0.47          7.0        11       23      ACTIVE
     Caleb Wilson  CHI ES

In [12]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8B — PER-PLAYER WEEKLY MOMENTS  (the engine's input)
# ═══════════════════════════════════════════════════════════════════
# The analytic engine (draft_app/engine.py) needs, per player: per-game means for the raw box-score
# quantities, per-game standard deviations for the counting categories, and the per-game standard
# deviation of each RATIO category's "impact" (FGM - rate*FGA, FTM - rate*FTA, PTS - rate*MIN), measured
# from real game logs so the FGM/FGA correlation is kept. Players with no log of their own (ESPN-only
# rookies) borrow their proxy's.
import sys; sys.path.insert(0, 'draft_app')
import engine, auction_sim as A
import importlib; importlib.reload(engine); importlib.reload(A)

F = P.copy()
for r in RAW:
    F['M_' + r] = F['P_' + r + '_M']
for c in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'DD']:
    F['S_' + c] = F['P_' + c + '_S']
_g = df_logs.assign(I_FG=df_logs.FGM - LEAGUE_RATE['FG_PCT'] * df_logs.FGA,
                    I_FT=df_logs.FTM - LEAGUE_RATE['FT_PCT'] * df_logs.FTA,
                    I_PPM=df_logs.PTS - LEAGUE_RATE['PTS_MIN'] * df_logs.MIN)
_sd = _g.groupby('PLAYER_ID')[['I_FG', 'I_FT', 'I_PPM']].std().rename(
    columns={'I_FG': 'SI_FG', 'I_FT': 'SI_FT', 'I_PPM': 'SI_PPM'})
F = F.merge(_sd, left_on=F['PROXY_PID'].fillna(F['PLAYER_ID']).astype(int), right_index=True, how='left')
for c in ['SI_FG', 'SI_FT', 'SI_PPM']:
    F[c] = F[c].fillna(F[c].median())
F['POS_ELIGIBLE_STR'] = F['POS_STR']
F = F.reset_index(drop=True)

RULES = engine.Rules(games_per_week=GAMES_PER_WEEK, cats_to_win=CATS_TO_WIN, roster_size=ROSTER_SIZE,
                     bench_count=BENCH_COUNT, budget=BUDGET_PER_TEAM, min_bid=MIN_BID, max_centers=MAX_CENTERS,
                     slots=tuple((n, tuple(a)) for n, a in STARTER_SLOTS))
MODEL = engine.build_pool_model(F, RULES, {k: LEAGUE_RATE[k] for k in ['FG_PCT', 'FT_PCT', 'PTS_MIN']})
ID2I = {pid: i for i, pid in enumerate(F['PLAYER_ID'])}
print(f'✅ engine pool model: {MODEL.n} players  ·  league rates FG% {LEAGUE_RATE["FG_PCT"]:.3f}, '
      f'FT% {LEAGUE_RATE["FT_PCT"]:.3f}, PPM {LEAGUE_RATE["PTS_MIN"]:.3f}')

✅ engine pool model: 382 players  ·  league rates FG% 0.473, FT% 0.785, PPM 0.486


## Cell 9 · Weekly simulator — a bootstrap of real games
Replaces v3's independent-normal simulator, whose errors are quantified below.

In [13]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — WEEKLY SIMULATOR: a bootstrap of REAL games   (replaces v3's)
# ═══════════════════════════════════════════════════════════════════
# v3's sim_weeks drew every box-score quantity as an INDEPENDENT normal. Free throws made and free throws
# attempted are then unrelated, so a team's weekly FT% = sum(FTM)/sum(FTA) swings wildly: that cell's own
# output has a weekly FT% standard deviation of ~0.18, against ~0.04 in real data. FG% was ~1.8x too noisy,
# and the clipped normals pushed double-double / steal / block means up by 3-15%.
# The consequence was not cosmetic: with FG% and FT% turned into permanent coin-flips, conceding them looked
# free and targeting them looked pointless. The bundle search then (correctly, given that simulator)
# concluded "punt FG% and FT%".
# This simulator resamples each player's actual games (jointly), rescaled to his projected means, so FGM
# moves with FGA, a double-double IS a game with two big numbers, and games played per week is
# Binomial(team games, availability).
GT = A.GameTables(df_logs, F, CURRENT_SEASON, PREV_SEASON, w_cur=W_CURRENT)

def sim_weeks_boot(lineup_idx, n_weeks, rng):
    """Weekly category totals for a lineup of row indices into F."""
    return A.bootstrap_weeks(F, GT, lineup_idx, n_weeks, rng, GAMES_PER_WEEK)

def sim_weeks_v3(lineup_idx, n_weeks, rng):
    """v3's simulator, kept ONLY to show what was wrong with it."""
    k = len(lineup_idx)
    avail = F['AVAIL'].to_numpy()[lineup_idx][None, :]
    g = rng.binomial(rng.binomial(4, GAMES_PER_WEEK/4.0, size=(n_weeks, k)), avail)
    tot = {}
    for c in RAW:
        mu = F['P_'+c+'_M'].to_numpy()[lineup_idx][None, :]; sd = F['P_'+c+'_S'].to_numpy()[lineup_idx][None, :]
        tot[c] = np.clip(g*mu + np.sqrt(g)*sd*rng.standard_normal((n_weeks, k)), 0, None).sum(axis=1)
    out = {c: tot[c] for c in COUNTING_CATS}; z = np.zeros(n_weeks)
    out['FG_PCT'] = np.divide(tot['FGM'], tot['FGA'], out=z.copy(), where=tot['FGA'] > 0)
    out['FT_PCT'] = np.divide(tot['FTM'], tot['FTA'], out=z.copy(), where=tot['FTA'] > 0)
    out['PTS_MIN'] = np.divide(tot['PTS'], tot['MIN'], out=z.copy(), where=tot['MIN'] > 0)
    return out

_rng = np.random.default_rng(RNG_SEED)
_top = np.argsort(-MODEL.E[:, CATS.index('PTS')])[:150]
_demo = list(_rng.choice(_top, 10, replace=False))
_new, _old = sim_weeks_boot(_demo, 30000, np.random.default_rng(1)), sim_weeks_v3(_demo, 30000, np.random.default_rng(1))
_mu, _sdv = MODEL.team_moments(_demo)
print('WEEKLY TEAM STANDARD DEVIATION — a random 10-man lineup, 30,000 simulated weeks')
print(f'  {"cat":>4}  {"analytic":>10} {"real-game bootstrap":>20} {"v3 simulator":>14}   {"v3 / real":>9}')
for ci, c in enumerate(CATS):
    r = _old[c].std() / _new[c].std()
    flag = '   <-- wrong' if abs(r - 1) > 0.25 else ''
    print(f'  {LBL[c]:>4}  {_sdv[ci]:10.3f} {_new[c].std():20.3f} {_old[c].std():14.3f}   {r:8.2f}x{flag}')
print('\nTHE SAME BIAS IN THE MEANS (v3 mean - bootstrap mean):  '
      + '  '.join(f'{LBL[c]} {_old[c].mean()-_new[c].mean():+.3g}' for c in ['DD', 'BLK', 'STL', 'FT_PCT']))

WEEKLY TEAM STANDARD DEVIATION — a random 10-man lineup, 30,000 simulated weeks
   cat    analytic  real-game bootstrap   v3 simulator   v3 / real
   FG%       0.030                0.030          0.060       1.97x   <-- wrong
   FT%       0.048                0.048          0.242       5.07x   <-- wrong
   3PM       9.006                8.947          8.890       0.99x
   REB      23.804               23.944         23.838       1.00x
   AST      15.219               15.661         15.229       0.97x
   STL       5.976                5.675          5.686       1.00x
   BLK       3.876                4.027          3.652       0.91x
    DD       1.691                1.718          1.558       0.91x
   PTS      63.562               63.096         63.399       1.00x
   PPM       0.048                0.047          0.060       1.29x   <-- wrong

THE SAME BIAS IN THE MEANS (v3 mean - bootstrap mean):  DD +0.494  BLK +0.744  STL +0.631  FT% +0.0328


## Cell 10 · Validating the analytic engine
The optimiser is only as good as the objective it climbs. 90 lineups are scored by both the engine and the simulator.

In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — VALIDATE THE ANALYTIC ENGINE AGAINST THE SIMULATOR
# ═══════════════════════════════════════════════════════════════════
# The engine evaluates a roster in microseconds, which is what makes optimising against the real objective
# (and doing it live during the draft) possible. Before trusting it: does it reproduce the simulator?
# Test set: 90 lineups (random good players, strong lineups, and lineups the optimiser itself produced),
# all scored against the same 13 opponent lineups.
_prices = P['MARKET_$'].to_numpy().astype(float)
_rival = [A.dollars_for(P, Z, KEY, strategy_value, auction_prices, p) for p in
          ['PTS+PPM+3PM+AST+FT%+STL', 'REB+BLK+DD+FG%+PTS+AST', 'AST+STL+3PM+PTS+FT%+PPM']]
_bal = A.dollars_for(P, Z, KEY, strategy_value, auction_prices, 'FG%+FT%+3PM+REB+AST+STL+BLK+DD+PTS+PPM')
WORLD = dict(model=MODEL, fld=None, pool=F, tables=GT, team_names=TEAM_NAMES, your_team=YOUR_TEAM_NAME,
             rival_plans=_rival)
# rough field to pick lineups with while the real benchmark is built
_top10 = np.argsort(-MODEL.E.sum(1))[:10]; _m0, _s0 = MODEL.team_moments(_top10)
WORLD['fld'] = engine.Field(mu=_m0*0.8, sd=_s0, tau=_s0*0.3)
def build_field(seeds, market):
    ros = []
    for s in seeds:
        r = A.run_trial(WORLD, 'plan', s, market, plan=_bal, n_weeks=100, want_sims=True)
        ros += [st for _, _, _, st in r['rosters'][1:]]
    return engine.Field.from_rosters(MODEL, ros), ros
FIELD, _opp = build_field([1, 2, 3, 4, 5], _prices)
WORLD['fld'] = FIELD
_opp_sims = [sim_weeks_boot(o, 8000, np.random.default_rng(300+i)) for i, o in enumerate(_opp[:13])]

def sim_result(lineup):
    mine = sim_weeks_boot(lineup, 8000, np.random.default_rng(99))
    return (float(np.mean([A.matchup_win_prob(mine, s) for s in _opp_sims])),
            float(np.mean([sum(A.cat_win_probs(mine, s).values()) for s in _opp_sims])))

_rng = np.random.default_rng(5); _rost = []
_good = np.argsort(-MODEL.E.sum(1))
for _ in range(35): _rost.append(list(_rng.choice(np.argsort(-MODEL.E[:, CATS.index('PTS')])[:150], 10, replace=False)))
for _ in range(25): _rost.append(list(_rng.choice(_good[:80], 10, replace=False)))
for _ in range(30):
    _pr = engine.Problem(MODEL, FIELD, np.maximum(1, np.round(_prices * _rng.uniform(0.7, 1.3, len(_prices)))),
                         F['MODEL_OK'].to_numpy(bool), budget=BUDGET_PER_TEAM)
    _o = engine.solve(_pr, restarts=(0.6,))
    if _o: _rost.append(list(_o[0]))
_rows = []
for ro in _rost:
    sp, se = sim_result(ro); ev = engine.evaluate(MODEL, ro, FIELD)
    ind = engine.matchup_prob(np.array(list(ev['p_cat'].values())), RULES, calibrate=False)
    _rows.append(dict(sim_p=sp, sim_e=se, eng_e=ev['e_cats'], eng_p=ev['p_win'], eng_p_raw=float(ind)))
V = pd.DataFrame(_rows)
lg = lambda p: np.log(np.clip(p, 1e-4, 1-1e-4) / (1 - np.clip(p, 1e-4, 1-1e-4)))
_x, _y = lg(V.eng_p_raw.to_numpy()), lg(V.sim_p.to_numpy())
_b = float((_x * _y).sum() / (_x * _x).sum()); _a = 0.0       # symmetric: a team equal to the field must be 50%
print(f'{len(V)} lineups scored by both')
print(f'  expected categories won : RMSE {np.sqrt(((V.eng_e-V.sim_e)**2).mean()):.3f}   correlation {V[["eng_e","sim_e"]].corr().iloc[0,1]:.4f}')
print(f'  P(win matchup), raw     : RMSE {np.sqrt(((V.eng_p_raw-V.sim_p)**2).mean()):.3f}   correlation {V[["eng_p_raw","sim_p"]].corr().iloc[0,1]:.4f}'
      f'   (categories treated as independent: too extreme)')
print(f'  P(win matchup), mapped  : RMSE {np.sqrt(((V.eng_p-V.sim_p)**2).mean()):.3f}   with  logit(sim) = {_b:.3f}*logit(raw)')
print(f'  the engine ships with  CAL_B={engine.CAL_B}   (re-fitted on this run: {_b:.3f})')
print('\n  Why the map exists: categories move together within a week, and 5-5 is a half win. Both are in the')
print('  bootstrap and neither is in a product of ten independent probabilities.')

90 lineups scored by both
  expected categories won : RMSE 0.113   correlation 0.9996
  P(win matchup), raw     : RMSE 0.085   correlation 0.9946   (categories treated as independent: too extreme)
  P(win matchup), mapped  : RMSE 0.016   with  logit(sim) = 0.608*logit(raw)
  the engine ships with  CAL_B=0.66   (re-fitted on this run: 0.608)

  Why the map exists: categories move together within a week, and 5-5 is a half win. Both are in the
  bootstrap and neither is in a product of ten independent probabilities.


## Cell 11 · The contested auction
The test v3 ran once. Four fixed category plans and the optimiser-driven bidder each play many full 14-team auctions, in three different market curves, and are scored on the weekly matchups against the rosters that came out of the same auction.

In [15]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — THE CONTESTED AUCTION: what actually wins when all 14 managers bid at once?
# ═══════════════════════════════════════════════════════════════════
# This is the test that matters, and v3 never ran it more than once. A simultaneous English auction
# (Cell 14 of v3, ported to draft_app/auction_sim.py): 14 budgeted bidders, rotating nominations, every
# player clearing at second-highest ceiling + $1, prices that move as money and talent leave the room.
# The rivals are market-following managers (3 of them run a punt plan, the rest bid the ESPN board with
# noise). "Me" is either
#   ENGINE : bids its BREAK-EVEN price from the optimiser: the most I can pay for this player and still end
#            with a roster as good as the best one I could build without him. Re-plans as the room moves.
#   a PLAN : a v3-style bidder with a fixed category plan (punt/target), valuing players by z-sum.
# Everyone is scored the same way: P(win a weekly matchup) against the 13 rosters that came out of the SAME
# auction, simulated with the real-game bootstrap.
N_TRIALS = 16                                   # auctions per (market, bidder); raise for tighter error bars
MARKETS = {k: P['MARKET_' + k].to_numpy().astype(float) for k in ['UNIFORM', 'STRUCTURAL', 'MODEL']}
PLANS = {'balanced, all 10 cats':           'FG%+FT%+3PM+REB+AST+STL+BLK+DD+PTS+PPM',
         '5-cat punt REB+AST+DD+PTS+PPM':   'REB+AST+DD+PTS+PPM',
         '7-cat 3PM+REB+AST+STL+BLK+DD+PTS': '3PM+REB+AST+STL+BLK+DD+PTS',
         'guard punt PTS+PPM+AST+STL+3PM+FT%': 'PTS+PPM+AST+STL+3PM+FT%'}
_dol = {n: A.dollars_for(P, Z, KEY, strategy_value, auction_prices, plan) for n, plan in PLANS.items()}
WORLD['rival_plans'] = [A.dollars_for(P, Z, KEY, strategy_value, auction_prices, p) for p in
                        ['PTS+PPM+3PM+AST+FT%+STL', 'REB+BLK+DD+FG%+PTS+AST', 'AST+STL+3PM+PTS+FT%+PPM']]
WORLD['model'], WORLD['pool'] = MODEL, F
rows, ENGINE_ROSTERS = [], {}
t0 = time.time()
for mname, mk in MARKETS.items():
    WORLD['fld'] = build_field([1, 2, 3, 4, 5], mk)[0]
    FIELD_BY_MARKET = globals().setdefault('FIELD_BY_MARKET', {}); FIELD_BY_MARKET[mname] = WORLD['fld']
    for bname in ['ENGINE'] + list(PLANS):
        res = []
        for s in range(N_TRIALS):
            if bname == 'ENGINE':
                r = A.run_trial(WORLD, 'engine', 1000 + s, mk, n_weeks=1200)
                ENGINE_ROSTERS.setdefault(mname, []).append((r['my_roster'], r['my_prices']))
            else:
                r = A.run_trial(WORLD, 'plan', 1000 + s, mk, plan=_dol[bname], n_weeks=1200)
            res.append((r['p_win'], r['e_cats'], r['spend']))
        a = np.array(res, float)
        rows.append({'market': mname, 'bidder': bname, 'P(win)': a[:, 0].mean(), '±': a[:, 0].std() / np.sqrt(len(a)),
                     'cats/wk': a[:, 1].mean(), 'spent $': a[:, 2].mean()})
    print(f'  {mname:<11} done  [{time.time()-t0:.0f}s]', flush=True)
MC = pd.DataFrame(rows)
print('\n' + '=' * 96)
print(f'  P(win a weekly matchup) in a CONTESTED auction   ({N_TRIALS} auctions per cell; ± = standard error)')
print('=' * 96)
for mname in MARKETS:
    d = MC[MC.market == mname].sort_values('P(win)', ascending=False)
    print(f'\n  market curve: {mname}')
    print(d[['bidder', 'P(win)', '±', 'cats/wk', 'spent $']].round(3).to_string(index=False))
_eng = MC[MC.bidder == 'ENGINE'].set_index('market')['P(win)']
_best_plan = MC[MC.bidder != 'ENGINE'].groupby('market')['P(win)'].max()
print('\n  ENGINE vs the best fixed plan:  ' + '   '.join(f'{m}: {_eng[m]:.3f} vs {_best_plan[m]:.3f} ({(_eng[m]-_best_plan[m])*100:+.0f} pts)' for m in MARKETS))

  UNIFORM     done  [62s]


  STRUCTURAL  done  [125s]


  MODEL       done  [189s]



  P(win a weekly matchup) in a CONTESTED auction   (16 auctions per cell; ± = standard error)

  market curve: UNIFORM
                            bidder  P(win)     ±  cats/wk  spent $
                            ENGINE   0.858 0.004    7.262  182.375
             balanced, all 10 cats   0.686 0.020    6.128  184.938
     5-cat punt REB+AST+DD+PTS+PPM   0.632 0.023    5.691  170.625
  7-cat 3PM+REB+AST+STL+BLK+DD+PTS   0.510 0.027    5.077  140.875
guard punt PTS+PPM+AST+STL+3PM+FT%   0.377 0.031    4.217  125.938

  market curve: STRUCTURAL
                            bidder  P(win)     ±  cats/wk  spent $
                            ENGINE   0.843 0.006    7.150  178.375
             balanced, all 10 cats   0.632 0.023    5.788  169.688
     5-cat punt REB+AST+DD+PTS+PPM   0.554 0.024    5.225  151.312
  7-cat 3PM+REB+AST+STL+BLK+DD+PTS   0.479 0.021    4.867  134.625
guard punt PTS+PPM+AST+STL+3PM+FT%   0.366 0.038    4.129  129.875

  market curve: MODEL
                         

## Cell 12 · Stress tests
Sharper rivals and a wrong market. A strategy that only beats passive opponents who share my price beliefs is not a strategy.

In [16]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — STRESS TESTS: does the edge survive sharper rivals and a wrong market?
# ═══════════════════════════════════════════════════════════════════
# An advantage measured against bots that follow the ESPN board proves little if it vanishes against managers
# who think. Two things are broken on purpose:
#   SHARP RIVALS   some of the 13 rivals run the SAME optimiser I do (each with its own noisy price beliefs).
#   WRONG MARKET   I believe one price curve, the room actually pays another. The tier adaptation (engine.live_prices)
#                  learns from the sales so far whether stars / mid-tier / cheap players clear above or below the
#                  board; without it only the overall price LEVEL adapts (money conservation). The comparison decides the
#                  default: if tiers do not clearly help, the assistant acts on the level only (use_tiers=False).
N_STRESS = 8
_mk_s, _mk_u = MARKETS['STRUCTURAL'], MARKETS['UNIFORM']
WORLD['fld'] = FIELD_BY_MARKET['STRUCTURAL']
def _run(label, n_runs=None, **kw):
    out = []
    for s in range(n_runs or N_STRESS):
        r = A.run_trial(WORLD, seed=3000 + s, n_weeks=1000, **kw); out.append((r['p_win'], r['e_cats'], r['spend']))
    a = np.array(out)
    print(f'  {label:<64} P(win) {a[:,0].mean():.3f} ±{a[:,0].std()/np.sqrt(len(a)):.3f}   cats/wk {a[:,1].mean():.2f}', flush=True)
    return a[:, 0].mean()
print('SHARP RIVALS  (structural market, me = engine)   — rivals running the optimiser are expensive to simulate, hence fewer auctions')
for k, n in ((0, N_STRESS), (3, 5)):
    _run(f'{k} of 13 rivals bid with the same optimiser', kind='engine', believed_market=_mk_s, n_sharp=k, n_runs=n)
print('\nWRONG MARKET  (me = engine)')
_run('room pays STRUCTURAL, I believe UNIFORM   | level only', kind='engine', believed_market=_mk_u, actual_market=_mk_s, use_tiers=False)
_run('room pays STRUCTURAL, I believe UNIFORM   | + tier adaptation', kind='engine', believed_market=_mk_u, actual_market=_mk_s, use_tiers=True)
_run('room pays UNIFORM,    I believe STRUCTURAL | level only', kind='engine', believed_market=_mk_s, actual_market=_mk_u, use_tiers=False)
_run('room pays UNIFORM,    I believe STRUCTURAL | + tier adaptation', kind='engine', believed_market=_mk_s, actual_market=_mk_u, use_tiers=True)

SHARP RIVALS  (structural market, me = engine)   — rivals running the optimiser are expensive to simulate, hence fewer auctions


  0 of 13 rivals bid with the same optimiser                       P(win) 0.858 ±0.008   cats/wk 7.26


  3 of 13 rivals bid with the same optimiser                       P(win) 0.725 ±0.007   cats/wk 6.39



WRONG MARKET  (me = engine)


  room pays STRUCTURAL, I believe UNIFORM   | level only           P(win) 0.860 ±0.008   cats/wk 7.25


  room pays STRUCTURAL, I believe UNIFORM   | + tier adaptation    P(win) 0.852 ±0.007   cats/wk 7.25


  room pays UNIFORM,    I believe STRUCTURAL | level only          P(win) 0.873 ±0.007   cats/wk 7.42


  room pays UNIFORM,    I believe STRUCTURAL | + tier adaptation   P(win) 0.867 ±0.007   cats/wk 7.36


np.float64(0.8670625000000001)

## Cell 13 · Target roster and break-even prices
The optimiser's roster at board prices is a sketch; the list of who the engine bidder actually ends up with across contested auctions is the realistic target list. Break-even prices say what each player is worth to a winning roster.

In [17]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — YOUR TARGET ROSTER, AND WHAT TO PAY
# ═══════════════════════════════════════════════════════════════════
# Two views, because the first is a sketch and the second is what a live auction produces:
#   (a) the optimiser's roster at the board prices, under each market curve (no competition: a best case);
#   (b) who the ENGINE bidder actually ends up with across the contested auctions of Cell 11, and what it pays.
# (b) is the honest target list; a name that appears in most auctions at a price near the board is a real target.
def show_roster(idx, price, title):
    ev = engine.evaluate(MODEL, idx, WORLD['fld'])
    d = pd.DataFrame({'PLAYER': [MODEL.names[i] for i in idx], 'POS': [F['POS_STR'].iat[i] for i in idx],
                      'TEAM': [F['TEAM'].iat[i] for i in idx], 'PRICE': [int(price[i]) for i in idx],
                      'AVAIL': [round(F['AVAIL'].iat[i], 2) for i in idx]})
    print(f'\n  {title}   (spend ${int(d.PRICE.sum())},  P(win matchup) {ev["p_win"]:.3f},  {ev["e_cats"]:.2f} cats/wk)')
    print(d.sort_values('PRICE', ascending=False).to_string(index=False))
    print('  category win probability: ' + '  '.join(f'{LBL[c]} {p:.2f}' for c, p in ev['p_cat'].items()))
    return ev
OK = F['MODEL_OK'].to_numpy(bool)
for mname in ['STRUCTURAL', 'UNIFORM']:
    WORLD['fld'] = FIELD_BY_MARKET[mname]
    pr = engine.Problem(MODEL, WORLD['fld'], np.maximum(MIN_BID, MARKETS[mname]), OK, budget=BUDGET_PER_TEAM)
    team, f, lam = engine.solve(pr)
    show_roster(team, MARKETS[mname], f'(a) optimiser roster, {mname} prices   [shadow price of money: {lam*100:.2f} pts of win prob per $10]')

print('\n' + '-' * 100)
print('  (b) WHO THE ENGINE BIDDER ENDS UP WITH — frequency across the contested auctions (STRUCTURAL market)')
print('-' * 100)
cnt = {}
for ros, prs in ENGINE_ROSTERS['STRUCTURAL']:
    for i, p in zip(ros, prs):
        c = cnt.setdefault(i, [0, 0.0]); c[0] += 1; c[1] += p
T = pd.DataFrame([{'PLAYER': MODEL.names[i], 'POS': F['POS_STR'].iat[i], 'TEAM': F['TEAM'].iat[i],
                   'ON MY ROSTER %': 100 * c / len(ENGINE_ROSTERS['STRUCTURAL']),
                   'AVG PAID $': s / c, 'BOARD $': MARKETS['STRUCTURAL'][i], 'MODEL $': F['MODEL_$'].iat[i]}
                  for i, (c, s) in cnt.items()]).sort_values('ON MY ROSTER %', ascending=False)
print(T.head(25).round(1).to_string(index=False))

# the break-even price of every player at the START of the draft (no one bought yet), board prices
WORLD['fld'] = FIELD_BY_MARKET['STRUCTURAL']
_pr0 = engine.Problem(MODEL, WORLD['fld'], np.maximum(MIN_BID, MARKETS['STRUCTURAL']), OK, budget=BUDGET_PER_TEAM)
_t0, _f0, _l0 = engine.solve(_pr0)
BREAK_EVEN = engine.max_bids(_pr0, _t0, max(_l0, 1e-4))
W = pd.DataFrame({'PLAYER': MODEL.names, 'POS': F['POS_STR'], 'TEAM': F['TEAM'], 'AVAIL': F['AVAIL'].round(2),
                  'BOARD $': MARKETS['STRUCTURAL'], 'PAY UP TO $': BREAK_EVEN.round(0)})
W['EDGE $'] = W['PAY UP TO $'] - W['BOARD $']; W = W[OK & (W['BOARD $'] > 0)]
print('\n  BUY — break-even price well above the board (stars excluded by price, not by name):')
print(W.nlargest(18, 'EDGE $').to_string(index=False))
print('\n  LET GO — the room will pay more than he is worth to a winning roster:')
print(W[W['BOARD $'] >= 8].nsmallest(14, 'EDGE $').to_string(index=False))


  (a) optimiser roster, STRUCTURAL prices   [shadow price of money: 0.09 pts of win prob per $10]   (spend $188,  P(win matchup) 0.842,  7.15 cats/wk)
            PLAYER  POS TEAM  PRICE  AVAIL
Karl-Anthony Towns    C  NYK     48   0.77
      Jamal Murray   PG  DEN     36   0.78
      Desmond Bane SGSF  ORL     19   0.80
     Chet Holmgren  PFC  OKC     19   0.71
   Trey Murphy III SGSF  NOP     16   0.73
     Julius Randle   PF  BKN     14   0.78
    Keyonte George PGSG  UTA     12   0.73
   Donovan Clingan    C  POR     12   0.81
       Kel'el Ware    C  MIL      9   0.79
    Brandon Ingram SFPF  LAC      3   0.74
  category win probability: FG% 0.55  FT% 0.70  3PM 0.75  REB 0.90  AST 0.63  STL 0.50  BLK 0.78  DD 0.80  PTS 0.83  PPM 0.72



  (a) optimiser roster, UNIFORM prices   [shadow price of money: 0.08 pts of win prob per $10]   (spend $197,  P(win matchup) 0.854,  7.23 cats/wk)
            PLAYER  POS TEAM  PRICE  AVAIL
Karl-Anthony Towns    C  NYK     50   0.77
      James Harden PGSG  CLE     37   0.74
      Jamal Murray   PG  DEN     37   0.78
      Desmond Bane SGSF  ORL     18   0.80
     Julius Randle   PF  BKN     13   0.78
       Joel Embiid    C  PHI     11   0.57
   Donovan Clingan    C  POR     11   0.81
    Keyonte George PGSG  UTA     10   0.73
       Kel'el Ware    C  MIL      8   0.79
    Brandon Ingram SFPF  LAC      2   0.74
  category win probability: FG% 0.47  FT% 0.76  3PM 0.72  REB 0.87  AST 0.77  STL 0.47  BLK 0.69  DD 0.83  PTS 0.85  PPM 0.80

----------------------------------------------------------------------------------------------------
  (b) WHO THE ENGINE BIDDER ENDS UP WITH — frequency across the contested auctions (STRUCTURAL market)
-----------------------------------------------


  BUY — break-even price well above the board (stars excluded by price, not by name):
            PLAYER  POS TEAM  AVAIL  BOARD $  PAY UP TO $  EDGE $
    Brandon Ingram SFPF  LAC   0.74      3.0         18.0    15.0
    Keyonte George PGSG  UTA   0.73     12.0         27.0    15.0
Karl-Anthony Towns    C  NYK   0.77     48.0         54.0     6.0
     Julius Randle   PF  BKN   0.78     14.0         19.0     5.0
       Kel'el Ware    C  MIL   0.79      9.0         14.0     5.0
      Jamal Murray   PG  DEN   0.78     36.0         40.0     4.0
      Desmond Bane SGSF  ORL   0.80     19.0         21.0     2.0
       Jalen Duren    C  DET   0.79     35.0         37.0     2.0
   Trey Murphy III SGSF  NOP   0.73     16.0         17.0     1.0
       Joel Embiid    C  PHI   0.57     12.0         12.0     0.0
    Onyeka Okongwu    C  ATL   0.77     11.0         11.0     0.0
     Chet Holmgren  PFC  OKC   0.71     19.0         19.0     0.0
        Al Horford  PFC  GSW   0.57      1.0          0

## Cell 14 · The season
One contested auction, then 18-matchup seasons played week by week for all 14 resulting rosters.

In [18]:
# ═══════════════════════════════════════════════════════════════════
# CELL 14 — THE SEASON: 18 matchups, record and playoff odds
# ═══════════════════════════════════════════════════════════════════
# One contested auction (the engine bidder at seat 'ME'), then 18-matchup seasons played week by week for all
# 14 rosters that came out of it. A week is decided by the real-game bootstrap; matchups are paired at random;
# 6 of 10 categories wins, 5-5 is half a win. Each team starts the best legal ten out of its 13.
N_SEASONS = 300
WORLD['fld'] = FIELD_BY_MARKET['STRUCTURAL']
_r = A.run_trial(WORLD, 'engine', 4242, MARKETS['STRUCTURAL'], n_weeks=100, want_sims=True)
_teams = _r['rosters']                                  # (name, roster, prices, starters)
def season_table(teams, n_seasons=N_SEASONS, seed=0):
    rng = np.random.default_rng(seed); n = len(teams); weeks = REGULAR_SEASON_MATCHUPS * n_seasons
    sims = [A.bootstrap_weeks(F, GT, st, weeks, np.random.default_rng(seed + 31*i), GAMES_PER_WEEK) for i, (_, _, _, st) in enumerate(teams)]
    wins = np.zeros((n_seasons, n)); catw = np.zeros((n_seasons, n))
    for s in range(n_seasons):
        for w in range(REGULAR_SEASON_MATCHUPS):
            idx = s*REGULAR_SEASON_MATCHUPS + w; order = rng.permutation(n)
            for a, b in zip(order[0::2], order[1::2]):
                ca = sum(1.0 if sims[a][c][idx] > sims[b][c][idx] else 0.5 if sims[a][c][idx] == sims[b][c][idx] else 0.0 for c in CATS)
                catw[s, a] += ca; catw[s, b] += len(CATS) - ca
                if ca > len(CATS)/2: wins[s, a] += 1
                elif ca < len(CATS)/2: wins[s, b] += 1
                else: wins[s, a] += .5; wins[s, b] += .5
    rank = (-wins).argsort(axis=1).argsort(axis=1)
    return pd.DataFrame({'TEAM': [t[0] for t in teams], 'W': wins.mean(0).round(1), 'L': (REGULAR_SEASON_MATCHUPS - wins.mean(0)).round(1),
                         'W_p10': np.percentile(wins, 10, axis=0).round(1), 'W_p90': np.percentile(wins, 90, axis=0).round(1),
                         'CATS/wk': (catw.mean(0) / REGULAR_SEASON_MATCHUPS).round(2),
                         'PLAYOFF%': ((rank < PLAYOFF_TEAMS).mean(0) * 100).round(1)}).sort_values('W', ascending=False)
SEASON_TABLE = season_table(_teams)
SEASON_TABLE['TEAM'] = SEASON_TABLE['TEAM'].replace({'ME': f'>>> {YOUR_TEAM_NAME}'})
print(SEASON_TABLE.to_string(index=False))
_me = SEASON_TABLE[SEASON_TABLE.TEAM.str.startswith('>>>')].iloc[0]
print(f"\n  {YOUR_TEAM_NAME}: {_me.W:.1f}-{_me.L:.1f}  ·  10th-90th percentile {_me.W_p10:.0f}-{REGULAR_SEASON_MATCHUPS-_me.W_p10:.0f} to "
      f"{_me.W_p90:.0f}-{REGULAR_SEASON_MATCHUPS-_me.W_p90:.0f}  ·  playoffs {_me['PLAYOFF%']:.0f}%")
print('  A caution on the absolute numbers: they assume the projections are right. The model has no allowance for')
print('  forecast error (the walk-forward backtest puts per-game production error at roughly half a cross-player sd).')
print('  Read them as "what happens if the room behaves like the simulated room"; the comparison between bidders is the finding.')

                      TEAM    W    L  W_p10  W_p90  CATS/wk  PLAYOFF%
       >>> Janartuo Ninjas 15.4  2.6   14.0   17.0     7.22      98.3
     Switchler to the moon 12.3  5.7    9.5   14.5     6.05      66.7
Transalpes Trail Shredders 11.8  6.2    9.5   14.1     5.72      62.0
          Mobiliar Mullets 11.5  6.5    9.0   14.0     5.82      51.7
                Brick City 11.3  6.7    9.0   13.5     5.66      54.0
    Load Management Slyrix 10.7  7.3    8.5   13.1     5.51      33.0
         Noah's Nasty Team 10.0  8.0    7.5   12.5     5.34      19.3
       Primitive Analytics  8.7  9.3    6.5   11.1     4.88      10.7
       Undersized Problems  7.5 10.5    5.0   10.0     4.43       1.7
                   Team 15  7.2 10.8    4.5    9.6     4.50       1.7
           Nordstern Hawks  6.7 11.3    4.5    9.0     4.26       0.7
                Ulifortada  5.0 13.0    3.0    7.5     3.70       0.3
        St. Wendelin Storm  5.0 13.0    3.0    7.5     3.81       0.0
          Inaugural 

## Cell 15 · Feed the live draft assistant
Writes `draft_app/player_pool.csv` and `league_meta.json`. The pool now carries the engine's per-player moments, the three market curves, the field benchmarks and the calibration constants; the app re-solves the optimiser after every logged pick. `draft_log.json` is never touched.

In [19]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15 — EXPORT FOR THE LIVE DRAFT ASSISTANT (draft_app/)
# ═══════════════════════════════════════════════════════════════════
import datetime, json as _json
APP_DIR = 'draft_app'
app = F.copy()
app['POS_PRIMARY'] = [sorted(s, key=ALL_POSITIONS.index)[0] for s in app['POS_ELIGIBLE']]
app['CURRENT_TEAM_ABBR'] = app['TEAM']; app['TEAMMATE_STRENGTH'] = app['MATES_DELTA'].round(3); app['TEAM_CONTEXT_FACTOR'] = app['CTX'].round(3)
app['RISK_FLAG'] = app['RISK']; app['AVAILABILITY'] = app['AVAIL'].round(3)
app['HEIGHT_TIER'] = pd.cut(app['HEIGHT_IN'], [0, 74, 77, 80, 99], labels=['small', 'wing', 'big wing', 'big']).astype(str)
_rookies = set(df_meta.loc[pd.to_numeric(df_meta['DRAFT_YEAR'], errors='coerce') >= DRAFT_DATE.year, 'PLAYER_ID'])
app['IS_ROOKIE'] = app['PLAYER_ID'].isin(_rookies) | (app['ROW_TYPE'] == 'ESPN-only (proxy line)')
for cat in COUNTING_CATS: app[f'PRED_{cat}_PG'] = app['P_' + cat + '_M'].round(3)
app['PRED_FG_PCT'] = np.where(app['P_FGA_M'] > 0, app['P_FGM_M'] / app['P_FGA_M'], np.nan).round(4)
app['PRED_FT_PCT'] = np.where(app['P_FTA_M'] > 0, app['P_FTM_M'] / app['P_FTA_M'], np.nan).round(4)
app['PRED_PTS_MIN'] = np.where(app['P_MIN_M'] > 0, app['P_PTS_M'] / app['P_MIN_M'], np.nan).round(4)
APP_COLS = (['PLAYER_ID', 'PLAYER_NAME', 'POS_ELIGIBLE_STR', 'POS_PRIMARY', 'AGE', 'HEIGHT_IN', 'HEIGHT_TIER', 'IS_ROOKIE',
             'CURRENT_TEAM_ABBR', 'TEAM_SOURCE', 'TEAMMATE_STRENGTH', 'TEAM_CONTEXT_FACTOR', 'RISK_FLAG', 'EXP_GP', 'AVAILABILITY',
             'AVAIL', 'ROW_TYPE', 'MODEL_OK', 'PROXY_PID', 'ON_ROSTER_2627', 'ESPN_$_RAW', 'ESPN_$_LIST', 'ESPN_INJURY', 'ESPN_PCT_OWNED', 'ESPN_RANK',
             'MARKET_$', 'MARKET_UNIFORM', 'MARKET_STRUCTURAL', 'MARKET_MODEL', 'MODEL_$', 'POS_SOURCE']
            + [f'PRED_{c}_PG' for c in COUNTING_CATS] + ['PRED_FG_PCT', 'PRED_FT_PCT', 'PRED_PTS_MIN']
            + ['M_' + r for r in RAW] + ['S_' + c for c in ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'DD']] + ['SI_FG', 'SI_FT', 'SI_PPM'])
# the legacy columns the pre-v4 app boards read stay as aliases so an old app.py still opens the file
app['EST_$BAL'] = app['MARKET_$']; app['EST_$PUNT'] = app['MARKET_$']
APP_COLS += ['EST_$BAL', 'EST_$PUNT']
out = app[APP_COLS].sort_values('MARKET_$', ascending=False)
os.makedirs(APP_DIR, exist_ok=True)
out.to_csv(f'{APP_DIR}/player_pool.csv', index=False)
_meta = {
    'league_name': LEAGUE_NAME, 'your_team_name': YOUR_TEAM_NAME, 'team_names': TEAM_NAMES, 'num_teams': NUM_TEAMS,
    'scoring_type': SCORING_TYPE, 'category_map': CATEGORY_MAP, 'draft_type': DRAFT_TYPE, 'draft_date': DRAFT_DATE.isoformat(),
    'budget_per_team': BUDGET_PER_TEAM, 'min_bid': MIN_BID, 'roster_size': ROSTER_SIZE, 'total_starters': TOTAL_STARTERS,
    'bench_count': BENCH_COUNT, 'ir_slots': IR_SLOTS, 'total_drafted': TOTAL_DRAFTED, 'max_centers': MAX_CENTERS,
    'position_maximums': POSITION_MAXIMUMS, 'starter_slots': [[n, list(a)] for n, a in STARTER_SLOTS],
    'lineup_changes': LINEUP_CHANGES, 'lineup_protection': LINEUP_PROTECTION, 'fab_budget': FAB_BUDGET,
    'matchup_acquisition_limit': MATCHUP_ACQUISITION_LIMIT, 'regular_season_matchups': REGULAR_SEASON_MATCHUPS,
    'playoff_teams': PLAYOFF_TEAMS, 'cats_to_win': CATS_TO_WIN, 'games_per_week': GAMES_PER_WEEK,
    # ── v4: what the live optimiser needs ──
    'model_version': 'v4-engine', 'market_model': MARKET_MODEL,
    'league_rates': {k: float(LEAGUE_RATE[k]) for k in ['FG_PCT', 'FT_PCT', 'PTS_MIN']},
    'field': {m: FIELD_BY_MARKET[m].to_dict() for m in FIELD_BY_MARKET},
    'calibration': {'cal_a': engine.CAL_A, 'cal_b': engine.CAL_B},
    'avail_coef': [float(x) for x in AVAIL_COEF],
    'generated_at': datetime.datetime.now().isoformat(timespec='seconds'),
}
with open(f'{APP_DIR}/league_meta.json', 'w') as f:
    _json.dump(_meta, f, indent=2, ensure_ascii=False)
print(f'✅ wrote {APP_DIR}/player_pool.csv  ({len(out)} players, {len(APP_COLS)} columns; '
      f'{int((~out.MODEL_OK).sum())} proxy-line players are loggable but never recommended)')
print(f'✅ wrote {APP_DIR}/league_meta.json (model {_meta["model_version"]}, board = {MARKET_MODEL})')
print(f'\n   Run it with:  streamlit run {APP_DIR}/app.py')

✅ wrote draft_app/player_pool.csv  (382 players, 65 columns; 6 proxy-line players are loggable but never recommended)
✅ wrote draft_app/league_meta.json (model v4-engine, board = STRUCTURAL)

   Run it with:  streamlit run draft_app/app.py
